In [1]:
!mkdir -p /kaggle/working/matcca

In [2]:
!cp /kaggle/input/datasets/siddhuuu101/matcca/*.py /kaggle/working/matcca/

In [1]:
%cd /kaggle/working/matcca

/kaggle/working/matcca


In [2]:
!ls

ablate.py		 paper_plots		tiny_transformer.py
attention.py		 plots			tokenizer.json
bpe_data.py		 profiling_results	train_real.py
checkpoints_scaled	 __pycache__		train_scaled.py
download_tinystories.py  real_data.py		train_tokenizer.py
extract_eigenvalues.py	 run_multiseed.py	water_filling.py
greedy_search.py	 state.db		water_filling_vs_greedy.py
inference_profile.py	 test_attention.py	water_filling_vs_greedy_real.py
make_paper_plots.py	 test_training.py	wf_vs_greedy_results.json
model_stats.csv		 tinystories_train.txt
model_stats_table.py	 tinystories_val.txt


In [3]:
import torch
print(torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

True Tesla T4


In [6]:
!python download_tinystories.py

This needs internet enabled in your Kaggle notebook's settings panel.
README.md: 1.06kB [00:00, 4.27MB/s]
data/train-00000-of-00004-2d5a1467fff108(…): 100%|█| 249M/249M [00:03<00:00, 82.
data/train-00001-of-00004-5852b56a2bd28f(…): 100%|█| 248M/248M [00:02<00:00, 112
data/train-00002-of-00004-a26307300439e9(…): 100%|█| 246M/246M [00:03<00:00, 81.
data/train-00003-of-00004-d243063613e5a0(…): 100%|█| 248M/248M [00:03<00:00, 77.
data/validation-00000-of-00001-869c898b5(…): 100%|█| 9.99M/9.99M [00:00<00:00, 2
Generating train split: 100%|█| 2119719/2119719 [00:06<00:00, 327339.10 examples
Generating validation split: 100%|█| 21990/21990 [00:00<00:00, 365608.33 example
Writing up to 200,000 training stories to 'tinystories_train.txt'...
Writing up to 2,000 validation stories to 'tinystories_val.txt'...

Done. 'tinystories_train.txt': 180.9 MB, 'tinystories_val.txt': 1.6 MB
Next: python train_tokenizer.py tinystories_train.txt tokenizer.json --vocab_size 8000


In [7]:
!python train_tokenizer.py tinystories_train.txt tokenizer.json --vocab_size 8000

Training BPE tokenizer, vocab_size=8000, on 'tinystories_train.txt'...
[00:00:00] Tokenize words                 ██████████████████ 33639    /    33639[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 33639    /    33639
[00:00:00] Compute merges                 ██████████████████ 7875     /     7875
Saved to 'tokenizer.json'. Actual vocab size: 8000

Round-trip sanity check on first 300 chars:
  original : 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with'...
  token IDs: [288, 224, 15, 127, 266, 316, 367, 227, 465, 127, 3747, 184, 178, 621, 17, 181, 573, 172, 150, 2836]... (75 tokens for 300 chars, compression ~4.00x chars/token)
  decoded  : 'One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with'...
  round-trip OK.


In [8]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  step     0 | train 329.8504 | val 246.5977 | 10s elapsed
  step   250 | train 7.2199 | val 7.0750 | 292s elapsed
  step   500 | train 6.8449 | val 6.6436 | 583s elapsed
  step   750 | train 4.8073 | val 4.7157 | 874s elapsed
  step  1000 | train 4.5122 | val 4.5245 | 1165s elapsed
  step  1250 | train 4.3667 | val 4.3750 | 1457s elapsed
  step  1500 | train 4.3631 | val 4.2862 | 1748s elapsed
  step  1750 | train 4.3960 | val 4.2211 | 2040s elapsed
  step  2000 | train 4.1193 | val 4.1076 | 2332s elapsed
  step  2250 | train 4.0719 | val 4.0311 | 2625s elapsed
  step  2500 | train 3.9641 | val 3.9481 | 2917s elapsed
  step  2750 | train 3.9552 | val 3.8834 | 3210s elapsed
  step  3000 | train 3.7922 | val 3.7896 | 3503s elapsed
  step  3250 | t

In [ ]:
!ls checkpoints_scaled/
!ls tinystories_train.txt tokenizer.json

In [9]:
!sed -i 's/SEEDS = \[0\]/SEEDS = [0, 1]/' train_scaled.py
!grep "SEEDS =" train_scaled.py

Runs single-seed by default (SEEDS = [0, 1]) — get this working and timed at real scale
SEEDS = [0, 1]              # add more seeds here once you've timed one pass


In [10]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.2 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.2 min

CCA fixed-rank (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.0962  |  total time: 0.2 min

MatCCA (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2019  |  total time: 0.2 min

MHA (seed=1)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3

In [7]:
!sed -i 's/SEEDS = \[0, 1\]/SEEDS = [0, 1, 2]/' train_scaled.py
!grep "SEEDS =" train_scaled.py

Runs single-seed by default (SEEDS = [0, 1, 2]) — get this working and timed at real scale
SEEDS = [0, 1, 2]              # add more seeds here once you've timed one pass


In [ ]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.1 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.1 min

CCA fixed-rank (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.0962  |  total time: 0.2 min

MatCCA (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2019  |  total time: 0.2 min

MHA (seed=1)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3

In [4]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.2 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.1 min

CCA fixed-rank (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.0962  |  total time: 0.2 min

MatCCA (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2019  |  total time: 0.2 min

MHA (seed=1)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3

In [5]:
%%writefile train_scaled.py

# Paste the entire contents of train_scaled.py below


"""
train_scaled.py — the real-scale run: real BPE-tokenized data (not char-level), a
config sized for a Kaggle T4 (16GB), full loss-curve tracking, matplotlib plots, and
cleanly formatted printed tables.

Default config here targets ~25-35M non-embedding params (dim=512, 8 layers, 8 heads,
seq_len=512) — an intermediate step up from the toy 4-5M scale, chosen to be safely
within a single Kaggle T4 session before pushing toward the full 50-150M target. If
you hit CUDA out-of-memory, first thing to reduce is BATCH_SIZE, not model size.

Runs single-seed by default (SEEDS = [0]) — get this working and timed at real scale
first, THEN decide if you want to spend the GPU-hours on multi-seed at this scale too
(edit SEEDS to add more once you've seen how long one pass takes).
"""

import os
import time
import json
import statistics
import torch
import matplotlib
matplotlib.use("Agg")  # no display needed, just save PNGs
import matplotlib.pyplot as plt

from tiny_transformer import TinyTransformerLM
from bpe_data import prepare_dataset, get_batch, estimate_loss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CORPUS_PATH = "tinystories_train.txt"       # from download_tinystories.py
TOKENIZER_PATH = "tokenizer.json"           # from train_tokenizer.py
CHECKPOINT_DIR = "checkpoints_scaled"
PLOTS_DIR = "plots"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ---- scaled-up config, sized for a Kaggle T4 (16GB) ----
DIM = 512
N_LAYERS = 8
N_HEADS = 8
SEQ_LEN = 512
BATCH_SIZE = 32          # reduce this first if you hit OOM, not DIM/N_LAYERS
STEPS = 5000
LR = 3e-4
EVAL_EVERY = 250
EVAL_BATCHES = 30
CHECKPOINT_EVERY = 1000
SEEDS = [0]              # add more seeds here once you've timed one pass


def train_one_variant(
    name, attn_kind, attn_kwargs, train_data, val_data, vocab_size,
    seed=0, sample_rank_fn=None, resume=True,
):
    print(f"\n{'=' * 70}\n{name} (seed={seed})\n{'=' * 70}")
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{name.replace(' ', '_')}_seed{seed}.pt")

    torch.manual_seed(seed)
    model = TinyTransformerLM(
        vocab_size=vocab_size, dim=DIM, n_layers=N_LAYERS, n_heads=N_HEADS,
        max_seq_len=SEQ_LEN, attn_kind=attn_kind, attn_kwargs=attn_kwargs,
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    start_step = 0
    history = {"step": [], "train_loss": [], "val_loss": []}

    if resume and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["optimizer"])
        start_step = ckpt["step"]
        history = ckpt.get("history", history)
        print(f"  resumed from checkpoint at step {start_step}")

    n_params = model.param_count()
    print(f"  params: {n_params:,}  |  device: {DEVICE}  |  seq_len: {SEQ_LEN}  |  batch: {BATCH_SIZE}")
    t0 = time.time()

    for step in range(start_step, STEPS):
        x, y = get_batch(train_data, BATCH_SIZE, SEQ_LEN, DEVICE)
        attn_kwargs_step = sample_rank_fn() if sample_rank_fn else {}
        _, loss = model(x, targets=y, **attn_kwargs_step)
        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % EVAL_EVERY == 0 or step == STEPS - 1:
            val_loss = estimate_loss(model, val_data, BATCH_SIZE, SEQ_LEN, DEVICE, EVAL_BATCHES, **attn_kwargs_step)
            elapsed = time.time() - t0
            history["step"].append(step)
            history["train_loss"].append(loss.item())
            history["val_loss"].append(val_loss)
            print(f"  step {step:5d} | train {loss.item():.4f} | val {val_loss:.4f} | {elapsed:.0f}s elapsed")

        if step % CHECKPOINT_EVERY == 0 and step > start_step:
            torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(),
                        "step": step, "history": history}, ckpt_path)

    torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(),
                "step": STEPS, "history": history}, ckpt_path)
    final_val = estimate_loss(model, val_data, BATCH_SIZE, SEQ_LEN, DEVICE, EVAL_BATCHES)
    total_time = time.time() - t0
    print(f"  FINAL val loss: {final_val:.4f}  |  total time: {total_time / 60:.1f} min")
    return model, final_val, history, n_params


def print_table(rows, headers):
    """Manually formatted table — aligned columns, no extra dependency."""
    widths = [max(len(str(h)), max((len(str(r[i])) for r in rows), default=0)) for i, h in enumerate(headers)]
    line = " | ".join(h.ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(str(c).ljust(w) for c, w in zip(r, widths)))


def plot_training_curves(all_histories, save_path):
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = {"MHA": "#4C72B0", "GQA": "#DD8452", "CCA fixed-rank": "#55A868", "MatCCA": "#C44E52"}
    for name, hist in all_histories.items():
        color = colors.get(name, None)
        ax.plot(hist["step"], hist["val_loss"], label=f"{name} (val)", color=color, linewidth=2)
    ax.set_xlabel("training step")
    ax.set_ylabel("validation loss")
    ax.set_title("Validation loss during training — real BPE-tokenized data")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def plot_final_comparison(summary, save_path):
    names = list(summary.keys())
    means = [summary[n][1] for n in names]
    stds = [summary[n][2] for n in names]
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(names, means, yerr=stds, capsize=6,
                   color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"][:len(names)])
    ax.set_ylabel("final validation loss (lower is better)")
    ax.set_title("Final val loss by architecture, real data")
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{mean:.3f}",
                ha="center", va="bottom")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def plot_matcca_curve(rank_results, save_path):
    fracs = sorted(rank_results.keys())
    means = [rank_results[f][1] for f in fracs]
    stds = [rank_results[f][2] for f in fracs]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(fracs, means, yerr=stds, marker="o", capsize=6, linewidth=2, color="#C44E52")
    ax.set_xlabel("active rank fraction (1.0 = uncompressed)")
    ax.set_ylabel("validation loss")
    ax.set_title("MatCCA: quality vs. compression level (one trained model)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def plot_nesting_cost(rank_stats, fixed_stats, save_path):
    """The plot that actually answers 'what does nesting cost you' — MatCCA's one-model
    curve against separately-trained fixed-rank models at the SAME compression levels,
    side by side. The gap between the two lines at each x-position IS the cost of nesting
    at that compression level, not just at one anchor point."""
    fracs = sorted(rank_stats.keys())
    matcca_means = [rank_stats[f][1] for f in fracs]
    matcca_stds = [rank_stats[f][2] for f in fracs]
    fixed_means = [fixed_stats[f][1] for f in fracs]
    fixed_stds = [fixed_stats[f][2] for f in fracs]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(fracs, matcca_means, yerr=matcca_stds, marker="o", capsize=6, linewidth=2,
                color="#C44E52", label="MatCCA (one nested model)")
    ax.errorbar(fracs, fixed_means, yerr=fixed_stds, marker="s", capsize=6, linewidth=2,
                color="#55A868", label="Separately-trained fixed-rank CCA")
    ax.set_xlabel("compression level (1.0 = compression=4, 0.5 = compression=8, 0.25 = compression=16)")
    ax.set_ylabel("validation loss")
    ax.set_title("The actual cost of nesting: MatCCA vs. dedicated models, at every level")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def main():
    print(f"Loading real BPE-tokenized data from '{CORPUS_PATH}' using '{TOKENIZER_PATH}'...")
    train_data, val_data, tok = prepare_dataset(CORPUS_PATH, TOKENIZER_PATH)
    train_data, val_data = train_data.to(DEVICE), val_data.to(DEVICE)
    print(f"vocab_size={tok.vocab_size}, train={len(train_data):,} tokens, "
          f"val={len(val_data):,} tokens, device={DEVICE}")

    common = dict(train_data=train_data, val_data=val_data, vocab_size=tok.vocab_size)
    summary, histories, rank_results, fixed_rank_results = {}, {}, {}, {}

    for seed in SEEDS:
        model, val_loss, hist, n_params = train_one_variant("MHA", "mha", {}, seed=seed, **common)
        summary.setdefault("MHA", []).append(val_loss)
        histories["MHA"] = hist

        _, val_loss, hist, _ = train_one_variant("GQA", "gqa", dict(n_kv_heads=2), seed=seed, **common)
        summary.setdefault("GQA", []).append(val_loss)
        histories["GQA"] = hist

        # Three SEPARATELY-TRAINED fixed-rank CCA models, one per compression level that
        # MatCCA's nested rank fractions correspond to (frac 1.0/0.5/0.25 <-> compression
        # 4/8/16). This is what actually completes the "cost of nesting" curve — before,
        # you only had the frac=1.0 anchor; now every point on MatCCA's curve has a
        # separately-trained baseline to compare against, not just the first one.
        fixed_rank_configs = {1.0: 4, 0.5: 8, 0.25: 16}
        for frac, compression in fixed_rank_configs.items():
            name = f"CCA fixed-rank (compression={compression})"
            _, val_loss, hist, _ = train_one_variant(
                name, "cca",
                dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=compression, compression_kv=compression),
                seed=seed, **common,
            )
            summary.setdefault(name, []).append(val_loss)
            histories[name] = hist
            fixed_rank_results.setdefault(frac, []).append(val_loss)

        max_r, head_dim = DIM // 4, (DIM // 4) // 8
        rank_fracs = [1.0, 0.5, 0.25]

        def sample_rank_fn():
            import random
            frac = random.choice(rank_fracs)
            r = max(head_dim, int(max_r * frac) - int(max_r * frac) % head_dim)
            return dict(active_rank_q=r, active_rank_kv=r)

        matcca_model, val_loss, hist, _ = train_one_variant(
            "MatCCA", "cca",
            dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=4, compression_kv=4),
            seed=seed, sample_rank_fn=sample_rank_fn, **common,
        )
        summary.setdefault("MatCCA", []).append(val_loss)
        histories["MatCCA"] = hist

        for frac in rank_fracs:
            r = max(head_dim, int(max_r * frac) - int(max_r * frac) % head_dim)
            rl = estimate_loss(matcca_model, val_data, BATCH_SIZE, SEQ_LEN, DEVICE, EVAL_BATCHES,
                                active_rank_q=r, active_rank_kv=r)
            rank_results.setdefault(frac, []).append(rl)

    # ---- printed summary table ----
    print(f"\n{'=' * 70}\nSUMMARY\n{'=' * 70}")
    rows = []
    stats = {}
    for name, vals in summary.items():
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if len(vals) > 1 else 0.0
        stats[name] = (vals, mean, std)
        rows.append([name, f"{mean:.4f}", f"{std:.4f}", str(len(vals))])
    print_table(rows, ["variant", "mean val loss", "std", "n seeds"])

    print("\nMatCCA by rank level (one model, evaluated at each compression):")
    rank_stats = {}
    rows2 = []
    for frac, vals in rank_results.items():
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if len(vals) > 1 else 0.0
        rank_stats[frac] = (vals, mean, std)
        rows2.append([f"{frac:.2f}", f"{mean:.4f}", f"{std:.4f}"])
    print_table(rows2, ["rank fraction", "mean val loss", "std"])

    print("\nSeparately-trained fixed-rank CCA, one model per compression level")
    print("(this is what MatCCA's curve above should be compared against — the actual")
    print("'cost of nesting' at every level, not just the frac=1.0 anchor):")
    fixed_stats = {}
    rows3 = []
    for frac, vals in fixed_rank_results.items():
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if len(vals) > 1 else 0.0
        fixed_stats[frac] = (vals, mean, std)
        rows3.append([f"{frac:.2f}", f"{mean:.4f}", f"{std:.4f}"])
    print_table(rows3, ["rank fraction", "mean val loss", "std"])

    # ---- plots ----
    print("\nGenerating plots...")
    plot_training_curves(histories, os.path.join(PLOTS_DIR, "training_curves.png"))
    plot_final_comparison(stats, os.path.join(PLOTS_DIR, "final_comparison.png"))
    plot_matcca_curve(rank_stats, os.path.join(PLOTS_DIR, "matcca_rank_curve.png"))
    plot_nesting_cost(rank_stats, fixed_stats, os.path.join(PLOTS_DIR, "nesting_cost_curve.png"))

    with open(os.path.join(PLOTS_DIR, "results_summary.json"), "w") as f:
        json.dump({
            "summary": {k: {"values": v[0], "mean": v[1], "std": v[2]} for k, v in stats.items()},
            "matcca_by_rank": {str(k): {"values": v[0], "mean": v[1], "std": v[2]} for k, v in rank_stats.items()},
            "fixed_rank_by_level": {str(k): {"values": v[0], "mean": v[1], "std": v[2]} for k, v in fixed_stats.items()},
        }, f, indent=2)
    print(f"\nAll done. Plots in '{PLOTS_DIR}/', raw numbers in '{PLOTS_DIR}/results_summary.json'.")


if __name__ == "__main__":
    main()

Overwriting train_scaled.py


In [ ]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.2 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.2 min

CCA fixed-rank (compression=4) (seed=0)
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  step     0 | train 327.0219 | val 259.9497 | 14s elapsed
  step   250 | train 7.2441 | val 7.1445 | 329s elapsed
  step   500 | train 5.1543 | val 5.1657 | 649s elapsed
  step   750 | train 4.7688 | val 4.6656 | 968s elapsed
  step  1000 | train 4.2550 | val 4.2894 | 1289s elapsed
  step  1250 | train 4.2914 | val 4.0986 | 1611s elapsed
  step  1500 | train 4.0454 | val 3.9467 | 193

In [4]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.2 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.1 min

CCA fixed-rank (compression=4) (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.0959  |  total time: 0.2 min

CCA fixed-rank (compression=8) (seed=0)
  resumed from checkpoint at step 2000
  params: 22,260,800  |  device: cuda  |  seq_len: 512  |  batch: 32
  step  2000 | train 3.7727 | val 3.8063 | 11s elapsed
  step  2250 | train 3.6786 | val 3.6990 | 250s elapsed
  step  2500 | train 3.6147 | val 3.613

In [4]:
%%writefile tiny_transformer.py
"""
tiny_transformer.py — a small decoder-only LM wrapping MHA / GQA / CCA interchangeably.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from attention import MultiHeadAttention, GroupedQueryAttention, CompressedConvolutionalAttention


class MLP(nn.Module):
    def __init__(self, dim, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * expansion),
            nn.GELU(),
            nn.Linear(dim * expansion, dim),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, dim, attn_module):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = attn_module
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim)

    def forward(self, x, **attn_kwargs):
        x = x + self.attn(self.norm1(x), **attn_kwargs)
        x = x + self.mlp(self.norm2(x))
        return x


def make_attention(kind, dim, n_heads, **kwargs):
    if kind == "mha":
        return MultiHeadAttention(dim, n_heads)
    if kind == "gqa":
        return GroupedQueryAttention(dim, n_heads, n_kv_heads=kwargs.get("n_kv_heads", n_heads // 4))
    if kind == "cca":
        return CompressedConvolutionalAttention(
            dim,
            n_heads_latent=kwargs.get("n_heads_latent", n_heads),
            n_kv_heads_latent=kwargs.get("n_kv_heads_latent", n_heads),
            compression_q=kwargs.get("compression_q", 4),
            compression_kv=kwargs.get("compression_kv", 4),
            conv_kernel=kwargs.get("conv_kernel", 4),
            use_conv_mix=kwargs.get("use_conv_mix", True),
            use_qk_mean=kwargs.get("use_qk_mean", True),
            use_v_shift=kwargs.get("use_v_shift", True),
        )
    raise ValueError(f"unknown attention kind: {kind}")


class TinyTransformerLM(nn.Module):
    def __init__(
        self, vocab_size, dim=128, n_layers=4, n_heads=8, max_seq_len=256,
        attn_kind="cca", attn_kwargs=None,
    ):
        super().__init__()
        attn_kwargs = attn_kwargs or {}
        self.max_seq_len = max_seq_len
        self.attn_kind = attn_kind
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([
            Block(dim, make_attention(attn_kind, dim, n_heads, **attn_kwargs))
            for _ in range(n_layers)
        ])
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(vocab_size, dim, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx, targets=None, per_layer_ranks=None, **attn_kwargs):
        B, T = idx.shape
        assert T <= self.max_seq_len
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        for i, block in enumerate(self.blocks):
            layer_kwargs = per_layer_ranks[i] if per_layer_ranks is not None else attn_kwargs
            x = block(x, **layer_kwargs)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def param_count(self):
        return sum(p.numel() for p in self.parameters())

Overwriting tiny_transformer.py


In [5]:
%%writefile greedy_search.py
"""
greedy_search.py — real MatryoshkaKV-style greedy search, adapted to CCA per-layer rank.
"""

import torch
import torch.nn.functional as F


@torch.no_grad()
def _kl_to_reference(model, batches, reference_logits_list, per_layer_ranks):
    model.eval()
    total_kl = 0.0
    for x, ref_logits in zip(batches, reference_logits_list):
        logits, _ = model(x, per_layer_ranks=per_layer_ranks)
        log_probs = F.log_softmax(logits, dim=-1)
        ref_probs = F.softmax(ref_logits, dim=-1)
        kl = F.kl_div(log_probs, ref_probs, reduction="batchmean")
        total_kl += kl.item()
    model.train()
    return total_kl / len(batches)


def greedy_rank_search(model, calib_batches, n_layers, head_dim, max_rank, target_total_rank,
                        verbose=True):
    xs = [x for x, _ in calib_batches]
    current_ranks = [max_rank] * n_layers
    min_rank = head_dim

    reference_cfg = [dict(active_rank_q=max_rank, active_rank_kv=max_rank)] * n_layers
    model.eval()
    with torch.no_grad():
        reference_logits_list = [model(x, per_layer_ranks=reference_cfg)[0].detach() for x in xs]
    model.train()

    n_forward_passes = len(xs)
    step = 0
    while sum(current_ranks) > target_total_rank:
        best_layer, best_kl, best_new_rank = None, float("inf"), None
        for layer_idx in range(n_layers):
            if current_ranks[layer_idx] <= min_rank:
                continue
            candidate = current_ranks.copy()
            candidate[layer_idx] = max(min_rank, candidate[layer_idx] - head_dim)
            per_layer_cfg = [dict(active_rank_q=r, active_rank_kv=r) for r in candidate]
            kl = _kl_to_reference(model, xs, reference_logits_list, per_layer_cfg)
            n_forward_passes += len(xs)
            if kl < best_kl:
                best_kl, best_layer, best_new_rank = kl, layer_idx, candidate[layer_idx]

        if best_layer is None:
            break
        current_ranks[best_layer] = best_new_rank
        step += 1
        if verbose:
            print(f"  greedy step {step}: reduced layer {best_layer} to rank {best_new_rank} "
                  f"(total={sum(current_ranks)}/{target_total_rank}, KL={best_kl:.5f}, "
                  f"forward passes so far={n_forward_passes})")

    return current_ranks, n_forward_passes

Writing greedy_search.py


In [6]:
%%writefile water_filling_vs_greedy.py
"""
water_filling_vs_greedy.py — compares water-filling vs real greedy on actual model forward passes.
"""

import time
import torch
import torch.nn.functional as F

from tiny_transformer import TinyTransformerLM
from water_filling import reverse_waterfilling_allocation
from greedy_search import greedy_rank_search


def extract_eigenvalues_per_layer(model, get_batch_fn, calib_batches=20, source="k"):
    ccas = [block.attn for block in model.blocks]
    for cca in ccas:
        cca.capture_activations = True
        cca._captured_k_lat.clear()
        cca._captured_q_lat.clear()

    model.eval()
    with torch.no_grad():
        for _ in range(calib_batches):
            x, _ = get_batch_fn()
            model(x)
    model.train()

    eigenvalues_per_layer = {}
    for i, cca in enumerate(ccas):
        captured = cca._captured_k_lat if source == "k" else cca._captured_q_lat
        A = torch.cat(captured, dim=0).reshape(-1, captured[0].shape[-1])
        A = A - A.mean(dim=0, keepdim=True)
        cov = (A.T @ A) / A.shape[0]
        eigvals = torch.linalg.eigvalsh(cov)
        eigvals = eigvals.flip(0).clamp(min=0).cpu().numpy()
        eigenvalues_per_layer[f"layer_{i}"] = eigvals
        cca.capture_activations = False

    return eigenvalues_per_layer


@torch.no_grad()
def evaluate_allocation(model, calib_batches, per_layer_ranks):
    model.eval()
    losses = []
    for x, y in calib_batches:
        _, loss = model(x, targets=y, per_layer_ranks=per_layer_ranks)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def allocation_dict_to_per_layer_ranks(allocation, n_layers, head_dim):
    ranks = []
    for i in range(n_layers):
        raw = allocation[f"layer_{i}"]
        rounded = max(head_dim, round(raw / head_dim) * head_dim)
        ranks.append(rounded)
    return [dict(active_rank_q=r, active_rank_kv=r) for r in ranks]


def run_comparison(model, get_batch_fn, calib_batches_for_eval, n_layers, head_dim, max_rank,
                    compression_factor=4, verbose=True):
    total_max_rank = n_layers * max_rank
    target_total_rank = total_max_rank // compression_factor
    target_total_rank = (target_total_rank // head_dim) * head_dim

    print(f"\n{'=' * 60}\nCompression factor {compression_factor}x "
          f"(target total rank: {target_total_rank}/{total_max_rank})\n{'=' * 60}")

    t0 = time.time()
    eigenvalues_per_layer = extract_eigenvalues_per_layer(model, get_batch_fn)
    wf_allocation, theta, distortion = reverse_waterfilling_allocation(
        eigenvalues_per_layer, target_total_rank)
    wf_time = time.time() - t0
    wf_per_layer_ranks = allocation_dict_to_per_layer_ranks(wf_allocation, n_layers, head_dim)
    wf_loss = evaluate_allocation(model, calib_batches_for_eval, wf_per_layer_ranks)
    print(f"water-filling: allocation={wf_allocation}")
    print(f"water-filling: wall-clock={wf_time:.3f}s, real calibration loss={wf_loss:.4f}")

    t0 = time.time()
    calib_for_greedy = [(x, y) for x, y in calib_batches_for_eval]
    greedy_ranks, n_forward_passes = greedy_rank_search(
        model, calib_for_greedy, n_layers, head_dim, max_rank, target_total_rank, verbose=verbose)
    greedy_time = time.time() - t0
    greedy_per_layer_ranks = [dict(active_rank_q=r, active_rank_kv=r) for r in greedy_ranks]
    greedy_loss = evaluate_allocation(model, calib_batches_for_eval, greedy_per_layer_ranks)
    print(f"greedy search: allocation={greedy_ranks}")
    print(f"greedy search: wall-clock={greedy_time:.3f}s, {n_forward_passes} forward passes, "
          f"real calibration loss={greedy_loss:.4f}")

    print(f"\n--- Verdict at {compression_factor}x ---")
    print(f"quality gap (water-filling - greedy): {wf_loss - greedy_loss:+.4f} "
          f"({'water-filling worse' if wf_loss > greedy_loss else 'water-filling better or equal'})")
    print(f"speed: water-filling is {greedy_time / max(wf_time, 1e-9):.1f}x faster, "
          f"{n_forward_passes}x fewer forward passes")

    return {
        "compression_factor": compression_factor,
        "water_filling": {"loss": wf_loss, "time_s": wf_time, "allocation": wf_allocation},
        "greedy": {"loss": greedy_loss, "time_s": greedy_time, "forward_passes": n_forward_passes,
                   "allocation": greedy_ranks},
    }

Overwriting water_filling_vs_greedy.py


In [11]:
%%writefile water_filling_vs_greedy_real.py
"""
water_filling_vs_greedy_real.py — Kaggle version: loads real trained MatCCA checkpoint.
"""

"""
water_filling_vs_greedy.py — the experiment that was always missing: does closed-form
water-filling actually match the real, expensive greedy search it's meant to replace?

Everything before this compared water-filling only against its own cheap distortion
proxy (guaranteed to agree by construction). This applies BOTH allocations back into
the actual trained model and measures REAL calibration loss/KL for each — the
comparison that was never run until now.

Two things get compared at the SAME total rank budget:
1. Quality: validation loss achieved by water-filling's allocation vs. greedy's allocation
2. Cost: a single sort (water-filling) vs. however many forward passes greedy needed
"""

import time
import torch
import torch.nn.functional as F

from tiny_transformer import TinyTransformerLM
from water_filling import reverse_waterfilling_allocation
from greedy_search import greedy_rank_search


def extract_eigenvalues_per_layer(model, get_batch_fn, calib_batches=20, source="k"):
    """Same mechanism as extract_eigenvalues.py, generalized to take any batch-sampling
    function so this works with real data (bpe_data.get_batch) as well as the toy task."""
    ccas = [block.attn for block in model.blocks]
    for cca in ccas:
        cca.capture_activations = True
        cca._captured_k_lat.clear()
        cca._captured_q_lat.clear()

    model.eval()
    with torch.no_grad():
        for _ in range(calib_batches):
            x, _ = get_batch_fn()
            model(x)
    model.train()

    eigenvalues_per_layer = {}
    for i, cca in enumerate(ccas):
        captured = cca._captured_k_lat if source == "k" else cca._captured_q_lat
        A = torch.cat(captured, dim=0).reshape(-1, captured[0].shape[-1])
        A = A - A.mean(dim=0, keepdim=True)
        cov = (A.T @ A) / A.shape[0]
        eigvals = torch.linalg.eigvalsh(cov)
        eigvals = eigvals.flip(0).clamp(min=0).cpu().numpy()
        eigenvalues_per_layer[f"layer_{i}"] = eigvals
        cca.capture_activations = False

    return eigenvalues_per_layer


@torch.no_grad()
def evaluate_allocation(model, calib_batches, per_layer_ranks):
    """Real validation loss achieved by a given per-layer rank allocation — this is
    what actually matters, not the internal distortion proxy either method optimizes
    for internally."""
    model.eval()
    losses = []
    for x, y in calib_batches:
        _, loss = model(x, targets=y, per_layer_ranks=per_layer_ranks)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def allocation_dict_to_per_layer_ranks(allocation, n_layers, head_dim):
    """water_filling.py allocates at individual-dimension granularity, but CCA can only
    truncate whole heads (active_rank must be a multiple of head_dim) — round each
    layer's raw allocation to the nearest valid multiple, minimum one head. This is a
    real, necessary adaptation, not a rounding nicety: without it, water-filling's
    output isn't a legal configuration for this architecture at all."""
    ranks = []
    for i in range(n_layers):
        raw = allocation[f"layer_{i}"]
        rounded = max(head_dim, round(raw / head_dim) * head_dim)
        ranks.append(rounded)
    return [dict(active_rank_q=r, active_rank_kv=r) for r in ranks]


def run_comparison(model, get_batch_fn, calib_batches_for_eval, n_layers, head_dim, max_rank,
                    compression_factor=4, verbose=True):
    """The full comparison at one target compression level."""
    total_max_rank = n_layers * max_rank
    target_total_rank = total_max_rank // compression_factor
    # round to nearest multiple of head_dim per layer on average
    target_total_rank = (target_total_rank // head_dim) * head_dim

    print(f"\n{'=' * 60}\nCompression factor {compression_factor}x "
          f"(target total rank: {target_total_rank}/{total_max_rank})\n{'=' * 60}")

    # ---- Water-filling: extract real eigenvalues, allocate, O(N log N) ----
    t0 = time.time()
    eigenvalues_per_layer = extract_eigenvalues_per_layer(model, get_batch_fn)
    wf_allocation, theta, distortion = reverse_waterfilling_allocation(
        eigenvalues_per_layer, target_total_rank)
    wf_time = time.time() - t0
    wf_per_layer_ranks = allocation_dict_to_per_layer_ranks(wf_allocation, n_layers, head_dim)
    wf_loss = evaluate_allocation(model, calib_batches_for_eval, wf_per_layer_ranks)
    print(f"water-filling: allocation={wf_allocation}")
    print(f"water-filling: wall-clock={wf_time:.3f}s, real calibration loss={wf_loss:.4f}")

    # ---- Real greedy search: expensive, iterative, KL-driven ----
    t0 = time.time()
    calib_for_greedy = [(x, y) for x, y in calib_batches_for_eval]
    greedy_ranks, n_forward_passes = greedy_rank_search(
        model, calib_for_greedy, n_layers, head_dim, max_rank, target_total_rank, verbose=verbose)
    greedy_time = time.time() - t0
    greedy_per_layer_ranks = [dict(active_rank_q=r, active_rank_kv=r) for r in greedy_ranks]
    greedy_loss = evaluate_allocation(model, calib_batches_for_eval, greedy_per_layer_ranks)
    print(f"greedy search: allocation={greedy_ranks}")
    print(f"greedy search: wall-clock={greedy_time:.3f}s, {n_forward_passes} forward passes, "
          f"real calibration loss={greedy_loss:.4f}")

    print(f"\n--- Verdict at {compression_factor}x ---")
    print(f"quality gap (water-filling loss - greedy loss): {wf_loss - greedy_loss:+.4f} "
          f"({'water-filling worse' if wf_loss > greedy_loss else 'water-filling better or equal'})")
    print(f"speed ratio: water-filling is {greedy_time / max(wf_time, 1e-9):.1f}x faster wall-clock, "
          f"{n_forward_passes}x fewer forward passes")

    return {
        "compression_factor": compression_factor,
        "water_filling": {"loss": wf_loss, "time_s": wf_time, "allocation": wf_allocation},
        "greedy": {"loss": greedy_loss, "time_s": greedy_time, "forward_passes": n_forward_passes,
                   "allocation": greedy_ranks},
    }


if __name__ == "__main__":
    # Self-contained smoke test on the toy periodic task, small scale — swap get_batch_fn
    # and the model for your real trained checkpoint + real data to run this for real.
    from test_training import make_periodic_batch

    torch.manual_seed(0)
    vocab_size, dim, n_layers, n_heads, seq_len = 64, 64, 4, 8, 32
    n_heads_latent, compression = 8, 4
    max_rank = dim // compression
    head_dim = max_rank // n_heads_latent

    model = TinyTransformerLM(
        vocab_size=vocab_size, dim=dim, n_layers=n_layers, n_heads=n_heads,
        max_seq_len=seq_len, attn_kind="cca",
        attn_kwargs=dict(n_heads_latent=n_heads_latent, n_kv_heads_latent=n_heads_latent,
                          compression_q=compression, compression_kv=compression),
    )

    print("Training a small MatCCA model with nested rank sampling first "
          "(need real trained weights before any of this means anything)...")
    import random
    opt = torch.optim.AdamW(model.parameters(), lr=1e-2)
    rank_fracs = [1.0, 0.5, 0.25]
    for step in range(400):
        frac = random.choice(rank_fracs)
        r = max(head_dim, int(max_rank * frac) - int(max_rank * frac) % head_dim)
        x, y = make_periodic_batch(16, seq_len, vocab_size)
        _, loss = model(x, targets=y, active_rank_q=r, active_rank_kv=r)
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"  trained, final loss={loss.item():.4f}")

    def get_batch_fn():
        x, y = make_periodic_batch(16, seq_len, vocab_size)
        return x, y

    calib_batches = [get_batch_fn() for _ in range(10)]

    results = []
    for compression_factor in [2, 4, 8]:
        r = run_comparison(model, get_batch_fn, calib_batches, n_layers, head_dim, max_rank,
                            compression_factor=compression_factor, verbose=False)
        results.append(r)

    print(f"\n{'=' * 60}\nSUMMARY\n{'=' * 60}")
    for r in results:
        wf, gr = r["water_filling"], r["greedy"]
        print(f"  {r['compression_factor']}x: water-filling loss={wf['loss']:.4f} "
              f"({wf['time_s']*1000:.1f}ms) vs greedy loss={gr['loss']:.4f} "
              f"({gr['forward_passes']} forward passes, {gr['time_s']:.2f}s)")

Overwriting water_filling_vs_greedy_real.py


In [10]:
!python water_filling_vs_greedy_real.py

Training a small MatCCA model with nested rank sampling first (need real trained weights before any of this means anything)...
  trained, final loss=1.0631

Compression factor 2x (target total rank: 32/64)
water-filling: allocation={'layer_0': 16, 'layer_1': 5, 'layer_2': 5, 'layer_3': 6}
water-filling: wall-clock=0.513s, real calibration loss=1.0726
greedy search: allocation=[16, 12, 2, 2]
greedy search: wall-clock=11.471s, 540 forward passes, real calibration loss=1.0549

--- Verdict at 2x ---
quality gap (water-filling loss - greedy loss): +0.0177 (water-filling worse)
speed ratio: water-filling is 22.4x faster wall-clock, 540x fewer forward passes

Compression factor 4x (target total rank: 16/64)
water-filling: allocation={'layer_0': 7, 'layer_1': 3, 'layer_2': 3, 'layer_3': 3}
water-filling: wall-clock=0.467s, real calibration loss=1.4690
greedy search: allocation=[10, 2, 2, 2]
greedy search: wall-clock=13.355s, 670 forward passes, real calibration loss=3.8386

--- Verdict at 4x -

In [3]:
%%writefile attention.py

"""
attention.py - MHA / GQA baselines plus Compressed Convolutional Attention (CCA/CCGQA),
with a Matryoshka-style truncation hook for the MatCCA experiment.

UPDATE: this now follows the paper's actual Listing 1 code and equations (8)-(11)
(Figliolia et al., arXiv:2510.04476), not a prose-based reconstruction. Three things
the earlier version got wrong, now fixed:
- QK-mean is a same-position average of PRE-conv q and k, added as a residual bias to
  POST-conv q/k -- NOT a causal running mean over time (that was a real conceptual bug,
  not just an approximation).
- "Channel mixing" is a second convolution, grouped by attention head (mixes channels
  within a head plus a small sequence window) -- NOT a dense LayerNorm+MLP block. The
  MLP version was almost certainly the main cause of CCA benchmarking ~1.5x SLOWER than
  MHA in this codebase, the opposite of the paper's claimed 1.7x speedup; the paper
  itself notes naive (non-fused) implementations of these ops carry real overhead, but
  a grouped conv should be far cheaper than a full MLP.
- Value-shift uses TWO SEPARATELY-LEARNED projections (one sees the current token, one
  sees the previous token), each producing half the heads -- NOT one projection reused
  with its output shifted in time.
- Normalization is L2-normalize-and-rescale with a learnable exponential key
  temperature -- NOT RMSNorm (close in spirit, not identical).
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class MultiHeadAttention(nn.Module):
    """Standard causal MHA -- baseline #1."""

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        self.qkv = nn.Linear(dim, 3 * dim, bias=False)
        self.out_proj = nn.Linear(dim, dim, bias=False)

    def forward(self, x, **kwargs):
        B, T, E = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, E)
        return self.out_proj(out)

    def param_count(self):
        return sum(p.numel() for p in self.parameters())


class GroupedQueryAttention(nn.Module):
    """Standard causal GQA -- baseline #2."""

    def __init__(self, dim: int, n_heads: int, n_kv_heads: int):
        super().__init__()
        assert dim % n_heads == 0 and n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = dim // n_heads
        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(dim, n_kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(n_heads * self.head_dim, dim, bias=False)

    def forward(self, x, **kwargs):
        B, T, E = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.head_dim).transpose(1, 2)
        rep = self.n_heads // self.n_kv_heads
        k = k.repeat_interleave(rep, dim=1)
        v = v.repeat_interleave(rep, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def kv_cache_dim_per_token(self):
        return 2 * self.n_kv_heads * self.head_dim

    def param_count(self):
        return sum(p.numel() for p in self.parameters())


def _causal_conv1d(x, weight, bias, groups, kernel_size):
    x_t = F.pad(x.transpose(1, 2), (kernel_size - 1, 0))
    out = F.conv1d(x_t, weight, bias, groups=groups)
    return out.transpose(1, 2)


class CCAConvMix(nn.Module):
    """Eq. (8): qtilde = conv2_seq+ch(conv1_seq(qtilde)). Two sequential causal convs:
    conv0 is fully depthwise (sequence-only mixing, no channel mixing at all).
    conv1 is grouped by attention head (mixes channels WITHIN a head, plus another
    small sequence window) -- replaces the dense MLP the earlier version used."""

    def __init__(self, latent_dim: int, n_heads: int, kernel_size: int = 4):
        super().__init__()
        assert latent_dim % n_heads == 0
        self.latent_dim = latent_dim
        self.n_heads = n_heads
        self.head_dim = latent_dim // n_heads
        self.kernel_size = kernel_size
        self.conv0 = nn.Conv1d(latent_dim, latent_dim, kernel_size, groups=latent_dim, bias=True)
        self.conv1 = nn.Conv1d(latent_dim, latent_dim, kernel_size, groups=n_heads, bias=True)

    def forward(self, x, active_dim: int = None):
        dim = active_dim or self.latent_dim
        active_heads = dim // self.head_dim

        w0 = self.conv0.weight[:dim]
        b0 = self.conv0.bias[:dim] if self.conv0.bias is not None else None
        out = _causal_conv1d(x, w0, b0, groups=dim, kernel_size=self.kernel_size)

        w1 = self.conv1.weight[:dim]
        b1 = self.conv1.bias[:dim] if self.conv1.bias is not None else None
        out = _causal_conv1d(out, w1, b1, groups=active_heads, kernel_size=self.kernel_size)
        return out

    def param_count(self):
        return sum(p.numel() for p in self.parameters())


def qk_mean_couple(q_pre, k_pre, q_post, k_post, n_q_heads, n_kv_heads):
    """Eq. (9): average the PRE-conv q and k at the same position (broadcasting k up
    to q's head count for GQA groups), then add that average as a residual bias to
    the POST-conv q and k. No time dimension involved."""
    rep = n_q_heads // n_kv_heads
    k_pre_b = k_pre.repeat_interleave(rep, dim=2) if rep > 1 else k_pre
    qk_mean_q = (q_pre + k_pre_b) / 2
    qk_mean_k = qk_mean_q.view(*qk_mean_q.shape[:2], n_kv_heads, rep, -1).mean(dim=3) if rep > 1 else qk_mean_q
    return q_post + qk_mean_q, k_post + qk_mean_k


def l2_norm_rescale(x, eps: float = 1e-6):
    head_dim = x.shape[-1]
    norm = x.norm(p=2, dim=-1, keepdim=True).clamp(min=eps)
    return x * (head_dim ** 0.5) / norm


class CompressedConvolutionalAttention(nn.Module):
    """CCA / CCGQA -- down-project Q, K, V into a shared compressed latent space and
    run the entire attention operation there. Supports Matryoshka-style truncation via
    active_rank_q / active_rank_kv for the MatCCA experiment."""

    def __init__(
        self, dim: int, n_heads_latent: int, n_kv_heads_latent: int = None,
        compression_q: int = 4, compression_kv: int = 4, conv_kernel: int = 4,
        use_conv_mix: bool = True, use_qk_mean: bool = True, use_v_shift: bool = True,
    ):
        super().__init__()
        n_kv_heads_latent = n_kv_heads_latent or n_heads_latent
        assert n_heads_latent % n_kv_heads_latent == 0
        self.dim = dim
        self.n_heads_latent = n_heads_latent
        self.n_kv_heads_latent = n_kv_heads_latent
        self.compression_q = compression_q
        self.compression_kv = compression_kv
        self.use_conv_mix = use_conv_mix
        self.use_qk_mean = use_qk_mean
        self.use_v_shift = use_v_shift

        self.latent_dim_q_max = dim // compression_q
        self.latent_dim_kv_max = dim // compression_kv
        assert self.latent_dim_q_max % n_heads_latent == 0
        assert self.latent_dim_kv_max % n_kv_heads_latent == 0
        assert self.latent_dim_kv_max % 2 == 0, "latent_dim_kv must be even (v-shift splits it in half)"
        self.head_dim_q = self.latent_dim_q_max // n_heads_latent
        self.head_dim_kv = self.latent_dim_kv_max // n_kv_heads_latent

        self.down_q = nn.Linear(dim, self.latent_dim_q_max, bias=False)
        self.down_k = nn.Linear(dim, self.latent_dim_kv_max, bias=False)
        half = self.latent_dim_kv_max // 2
        self.down_v_now = nn.Linear(dim, half, bias=False)
        self.down_v_prev = nn.Linear(dim, self.latent_dim_kv_max - half, bias=False)

        self.conv_q = CCAConvMix(self.latent_dim_q_max, n_heads_latent, conv_kernel)
        self.conv_k = CCAConvMix(self.latent_dim_kv_max, n_kv_heads_latent, conv_kernel)

        self.key_temp = nn.Parameter(torch.zeros(n_kv_heads_latent))
        self.up_proj = nn.Linear(self.latent_dim_q_max, dim, bias=False)

        self.capture_activations = False
        self._captured_k_lat = []
        self._captured_q_lat = []

    def forward(self, x, active_rank_q: int = None, active_rank_kv: int = None):
        B, T, E = x.shape

        r_q = active_rank_q or self.latent_dim_q_max
        r_kv = active_rank_kv or self.latent_dim_kv_max
        assert r_q % self.head_dim_q == 0, "active_rank_q must be a multiple of head_dim_q"
        assert r_kv % self.head_dim_kv == 0, "active_rank_kv must be a multiple of head_dim_kv"
        assert r_kv % 2 == 0, "active_rank_kv must stay even (v-shift splits it in half)"
        h_q = r_q // self.head_dim_q
        h_kv = r_kv // self.head_dim_kv

        q_pre = F.linear(x, self.down_q.weight[:r_q, :])
        k_pre = F.linear(x, self.down_k.weight[:r_kv, :])

        if self.capture_activations:
            self._captured_q_lat.append(q_pre.detach())
            self._captured_k_lat.append(k_pre.detach())

        if self.use_conv_mix:
            q_post = self.conv_q(q_pre, active_dim=r_q)
            k_post = self.conv_k(k_pre, active_dim=r_kv)
        else:
            q_post, k_post = q_pre, k_pre

        q_pre_h = q_pre.view(B, T, h_q, self.head_dim_q)
        k_pre_h = k_pre.view(B, T, h_kv, self.head_dim_kv)
        q_post_h = q_post.view(B, T, h_q, self.head_dim_q)
        k_post_h = k_post.view(B, T, h_kv, self.head_dim_kv)

        if self.use_qk_mean:
            q, k = qk_mean_couple(q_pre_h, k_pre_h, q_post_h, k_post_h, h_q, h_kv)
        else:
            q, k = q_post_h, k_post_h

        q = l2_norm_rescale(q)
        k = l2_norm_rescale(k) * torch.exp(self.key_temp[:h_kv]).view(1, 1, h_kv, 1)

        r_half = r_kv // 2
        if self.use_v_shift:
            x_prev = F.pad(x[:, :-1, :], (0, 0, 1, 0))
            v_now = F.linear(x, self.down_v_now.weight[:r_half, :])
            v_prev = F.linear(x_prev, self.down_v_prev.weight[:r_half, :])
            v_lat = torch.cat([v_now, v_prev], dim=-1)
        else:
            v_now = F.linear(x, self.down_v_now.weight[:r_half, :])
            v_prev = F.linear(x, self.down_v_prev.weight[:r_half, :])
            v_lat = torch.cat([v_now, v_prev], dim=-1)
        v = v_lat.view(B, T, h_kv, self.head_dim_kv)

        q, k, v = (t.transpose(1, 2) for t in (q, k, v))
        if h_q != h_kv:
            assert h_q % h_kv == 0, "active latent query/kv heads must be GQA-compatible"
            rep = h_q // h_kv
            k = k.repeat_interleave(rep, dim=1)
            v = v.repeat_interleave(rep, dim=1)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, r_q)
        out = F.linear(out, self.up_proj.weight[:, :r_q])
        return out

    def kv_cache_dim_per_token(self, active_rank_kv: int = None):
        r_kv = active_rank_kv or self.latent_dim_kv_max
        return 2 * r_kv

    def param_count(self):
        return sum(p.numel() for p in self.parameters())


class MultiHeadLatentAttention(nn.Module):
    """MLA — Multi-head Latent Attention from DeepSeek-V2 (arXiv:2405.04434).

    The key idea: instead of caching full-size K and V per head (MHA) or sharing
    K/V heads (GQA), MLA compresses K and V into a single shared low-rank latent
    vector and caches THAT — so the KV-cache shrinks by the compression factor
    while keeping close to MHA's full per-head expressiveness.

    This is the closest prior method to CCA in the literature, and the one CCA's
    own paper benchmarks against — so including it here is the comparison the
    reviewer report said was missing. Critical difference from CCA: MLA's latent
    only compresses K/V (queries stay full-size and full-cost), so FLOPs are
    unchanged. CCA compresses Q, K, and V together and attends in the compressed
    space, so FLOPs shrink too.

    Implementation follows DeepSeek-V2's equations directly:
    - c_KV = W_DKV * x              (down-project input to shared KV latent)
    - k_C = W_UK * c_KV             (up-project latent to compressed keys)
    - v_C = W_UV * c_KV             (up-project latent to compressed values)
    - q = W_Q * x                   (queries are full-size, no compression)
    - attention in full Q/K/V space, cache only c_KV at inference
    """

    def __init__(self, dim: int, n_heads: int, kv_lora_rank: int = None):
        super().__init__()
        assert dim % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = dim // n_heads
        # kv_lora_rank is the compressed latent dimension for K/V.
        # DeepSeek-V2 uses 512 for a 5120-dim model (10x compression).
        # We default to dim//4 to match the compression factor used for
        # CCA/CCGQA in this experiment, making KV-cache sizes comparable.
        self.kv_lora_rank = kv_lora_rank or (dim // 4)

        # Q projection (full-size — no compression on queries in MLA)
        self.q_proj = nn.Linear(dim, n_heads * self.head_dim, bias=False)

        # KV down-projection: input -> shared low-rank latent (cached at inference)
        self.kv_down_proj = nn.Linear(dim, self.kv_lora_rank, bias=False)

        # KV up-projections: latent -> full K and V (computed from cache at inference)
        self.k_up_proj = nn.Linear(self.kv_lora_rank, n_heads * self.head_dim, bias=False)
        self.v_up_proj = nn.Linear(self.kv_lora_rank, n_heads * self.head_dim, bias=False)

        self.out_proj = nn.Linear(n_heads * self.head_dim, dim, bias=False)

    def forward(self, x, **kwargs):
        B, T, E = x.shape

        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # Compress to latent, then up-project to full K and V
        c_kv = self.kv_down_proj(x)                          # (B, T, kv_lora_rank)
        k = self.k_up_proj(c_kv).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_up_proj(c_kv).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        out = out.transpose(1, 2).reshape(B, T, self.n_heads * self.head_dim)
        return self.out_proj(out)

    def kv_cache_dim_per_token(self):
        # At inference, only the latent c_KV is cached — not full K and V.
        return self.kv_lora_rank

    def param_count(self):
        return sum(p.numel() for p in self.parameters())


Overwriting attention.py


In [4]:
%%writefile tiny_transformer.py

"""
tiny_transformer.py — a small decoder-only LM wrapping MHA / GQA / CCA interchangeably.

Default config is deliberately tiny (a few million params) so this trains fast on CPU
for correctness checks here; bump dim/n_layers/vocab up for a real Kaggle T4 run (see
the scoping doc's suggested plan: 50-150M params, C4-scale data, matched token budget
across conditions).

Positional encoding: plain learned absolute embeddings, for simplicity in this first
scaffold. Real CCA/MLA-style models use RoPE; integrating RoPE into the compressed
latent space is its own subtlety (where exactly it composes with the down-projection)
that isn't nailed down here — flagged as a follow-up, not done yet.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F

from attention import MultiHeadAttention, GroupedQueryAttention, CompressedConvolutionalAttention, MultiHeadLatentAttention


class MLP(nn.Module):
    def __init__(self, dim, expansion=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * expansion),
            nn.GELU(),
            nn.Linear(dim * expansion, dim),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, dim, attn_module):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = attn_module
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = MLP(dim)

    def forward(self, x, **attn_kwargs):
        x = x + self.attn(self.norm1(x), **attn_kwargs)
        x = x + self.mlp(self.norm2(x))
        return x


def make_attention(kind, dim, n_heads, **kwargs):
    if kind == "mha":
        return MultiHeadAttention(dim, n_heads)
    if kind == "gqa":
        return GroupedQueryAttention(dim, n_heads, n_kv_heads=kwargs.get("n_kv_heads", n_heads // 4))
    if kind == "mla":
        return MultiHeadLatentAttention(
            dim, n_heads,
            kv_lora_rank=kwargs.get("kv_lora_rank", dim // 4),
        )
    if kind == "cca":
        return CompressedConvolutionalAttention(
            dim,
            n_heads_latent=kwargs.get("n_heads_latent", n_heads),
            n_kv_heads_latent=kwargs.get("n_kv_heads_latent", n_heads),
            compression_q=kwargs.get("compression_q", 4),
            compression_kv=kwargs.get("compression_kv", 4),
            conv_kernel=kwargs.get("conv_kernel", 4),
            use_conv_mix=kwargs.get("use_conv_mix", True),
            use_qk_mean=kwargs.get("use_qk_mean", True),
            use_v_shift=kwargs.get("use_v_shift", True),
        )
    raise ValueError(f"unknown attention kind: {kind}")


class TinyTransformerLM(nn.Module):
    def __init__(
        self, vocab_size, dim=128, n_layers=4, n_heads=8, max_seq_len=256,
        attn_kind="cca", attn_kwargs=None,
    ):
        super().__init__()
        attn_kwargs = attn_kwargs or {}
        self.max_seq_len = max_seq_len
        self.attn_kind = attn_kind
        self.tok_emb = nn.Embedding(vocab_size, dim)
        self.pos_emb = nn.Embedding(max_seq_len, dim)
        self.blocks = nn.ModuleList([
            Block(dim, make_attention(attn_kind, dim, n_heads, **attn_kwargs))
            for _ in range(n_layers)
        ])
        self.norm_f = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx, targets=None, per_layer_ranks=None, **attn_kwargs):
        """per_layer_ranks: optional list of dicts, one per layer, e.g.
        [{'active_rank_q': 32, 'active_rank_kv': 32}, {'active_rank_q': 16, ...}, ...] —
        lets each layer run at a DIFFERENT rank, which is what water-filling and greedy
        search both actually produce (a heterogeneous per-layer map, not one global
        number). If not given, falls back to the old behavior: the same attn_kwargs
        applied uniformly to every layer.
        """
        B, T = idx.shape
        assert T <= self.max_seq_len
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)[None, :, :]
        for i, block in enumerate(self.blocks):
            layer_kwargs = per_layer_ranks[i] if per_layer_ranks is not None else attn_kwargs
            x = block(x, **layer_kwargs)
        x = self.norm_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss

    def param_count(self):
        return sum(p.numel() for p in self.parameters())

Overwriting tiny_transformer.py


In [5]:
%%writefile train_scaled.py

"""
train_scaled.py — the real-scale run: real BPE-tokenized data (not char-level), a
config sized for a Kaggle T4 (16GB), full loss-curve tracking, matplotlib plots, and
cleanly formatted printed tables.

Default config here targets ~25-35M non-embedding params (dim=512, 8 layers, 8 heads,
seq_len=512) — an intermediate step up from the toy 4-5M scale, chosen to be safely
within a single Kaggle T4 session before pushing toward the full 50-150M target. If
you hit CUDA out-of-memory, first thing to reduce is BATCH_SIZE, not model size.

Runs single-seed by default (SEEDS = [0]) — get this working and timed at real scale
first, THEN decide if you want to spend the GPU-hours on multi-seed at this scale too
(edit SEEDS to add more once you've seen how long one pass takes).
"""

import os
import time
import json
import statistics
import torch
import matplotlib
matplotlib.use("Agg")  # no display needed, just save PNGs
import matplotlib.pyplot as plt

from tiny_transformer import TinyTransformerLM
from bpe_data import prepare_dataset, get_batch, estimate_loss

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CORPUS_PATH = "tinystories_train.txt"       # from download_tinystories.py
TOKENIZER_PATH = "tokenizer.json"           # from train_tokenizer.py
CHECKPOINT_DIR = "checkpoints_scaled"
PLOTS_DIR = "plots"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ---- scaled-up config, sized for a Kaggle T4 (16GB) ----
DIM = 512
N_LAYERS = 8
N_HEADS = 8
SEQ_LEN = 512
BATCH_SIZE = 32          # reduce this first if you hit OOM, not DIM/N_LAYERS
STEPS = 5000
LR = 3e-4
EVAL_EVERY = 250
EVAL_BATCHES = 30
CHECKPOINT_EVERY = 1000
SEEDS = [0]              # add more seeds here once you've timed one pass


def train_one_variant(
    name, attn_kind, attn_kwargs, train_data, val_data, vocab_size,
    seed=0, sample_rank_fn=None, resume=True,
):
    print(f"\n{'=' * 70}\n{name} (seed={seed})\n{'=' * 70}")
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{name.replace(' ', '_')}_seed{seed}.pt")

    torch.manual_seed(seed)
    model = TinyTransformerLM(
        vocab_size=vocab_size, dim=DIM, n_layers=N_LAYERS, n_heads=N_HEADS,
        max_seq_len=SEQ_LEN, attn_kind=attn_kind, attn_kwargs=attn_kwargs,
    ).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    start_step = 0
    history = {"step": [], "train_loss": [], "val_loss": []}

    if resume and os.path.exists(ckpt_path):
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        opt.load_state_dict(ckpt["optimizer"])
        start_step = ckpt["step"]
        history = ckpt.get("history", history)
        print(f"  resumed from checkpoint at step {start_step}")

    n_params = model.param_count()
    print(f"  params: {n_params:,}  |  device: {DEVICE}  |  seq_len: {SEQ_LEN}  |  batch: {BATCH_SIZE}")
    t0 = time.time()

    for step in range(start_step, STEPS):
        x, y = get_batch(train_data, BATCH_SIZE, SEQ_LEN, DEVICE)
        attn_kwargs_step = sample_rank_fn() if sample_rank_fn else {}
        _, loss = model(x, targets=y, **attn_kwargs_step)
        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % EVAL_EVERY == 0 or step == STEPS - 1:
            val_loss = estimate_loss(model, val_data, BATCH_SIZE, SEQ_LEN, DEVICE, EVAL_BATCHES, **attn_kwargs_step)
            elapsed = time.time() - t0
            history["step"].append(step)
            history["train_loss"].append(loss.item())
            history["val_loss"].append(val_loss)
            print(f"  step {step:5d} | train {loss.item():.4f} | val {val_loss:.4f} | {elapsed:.0f}s elapsed")

        if step % CHECKPOINT_EVERY == 0 and step > start_step:
            torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(),
                        "step": step, "history": history}, ckpt_path)

    torch.save({"model": model.state_dict(), "optimizer": opt.state_dict(),
                "step": STEPS, "history": history}, ckpt_path)
    final_val = estimate_loss(model, val_data, BATCH_SIZE, SEQ_LEN, DEVICE, EVAL_BATCHES)
    total_time = time.time() - t0
    print(f"  FINAL val loss: {final_val:.4f}  |  total time: {total_time / 60:.1f} min")
    return model, final_val, history, n_params


def print_table(rows, headers):
    """Manually formatted table — aligned columns, no extra dependency."""
    widths = [max(len(str(h)), max((len(str(r[i])) for r in rows), default=0)) for i, h in enumerate(headers)]
    line = " | ".join(h.ljust(w) for h, w in zip(headers, widths))
    print(line)
    print("-" * len(line))
    for r in rows:
        print(" | ".join(str(c).ljust(w) for c, w in zip(r, widths)))


def plot_training_curves(all_histories, save_path):
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = {"MHA": "#4C72B0", "GQA": "#DD8452", "CCA fixed-rank": "#55A868", "MatCCA": "#C44E52"}
    for name, hist in all_histories.items():
        color = colors.get(name, None)
        ax.plot(hist["step"], hist["val_loss"], label=f"{name} (val)", color=color, linewidth=2)
    ax.set_xlabel("training step")
    ax.set_ylabel("validation loss")
    ax.set_title("Validation loss during training — real BPE-tokenized data")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def plot_final_comparison(summary, save_path):
    names = list(summary.keys())
    means = [summary[n][1] for n in names]
    stds = [summary[n][2] for n in names]
    fig, ax = plt.subplots(figsize=(7, 5))
    bars = ax.bar(names, means, yerr=stds, capsize=6,
                   color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"][:len(names)])
    ax.set_ylabel("final validation loss (lower is better)")
    ax.set_title("Final val loss by architecture, real data")
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{mean:.3f}",
                ha="center", va="bottom")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def plot_matcca_curve(rank_results, save_path):
    fracs = sorted(rank_results.keys())
    means = [rank_results[f][1] for f in fracs]
    stds = [rank_results[f][2] for f in fracs]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(fracs, means, yerr=stds, marker="o", capsize=6, linewidth=2, color="#C44E52")
    ax.set_xlabel("active rank fraction (1.0 = uncompressed)")
    ax.set_ylabel("validation loss")
    ax.set_title("MatCCA: quality vs. compression level (one trained model)")
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def plot_nesting_cost(rank_stats, fixed_stats, save_path):
    """The plot that actually answers 'what does nesting cost you' — MatCCA's one-model
    curve against separately-trained fixed-rank models at the SAME compression levels,
    side by side. The gap between the two lines at each x-position IS the cost of nesting
    at that compression level, not just at one anchor point."""
    fracs = sorted(rank_stats.keys())
    matcca_means = [rank_stats[f][1] for f in fracs]
    matcca_stds = [rank_stats[f][2] for f in fracs]
    fixed_means = [fixed_stats[f][1] for f in fracs]
    fixed_stds = [fixed_stats[f][2] for f in fracs]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(fracs, matcca_means, yerr=matcca_stds, marker="o", capsize=6, linewidth=2,
                color="#C44E52", label="MatCCA (one nested model)")
    ax.errorbar(fracs, fixed_means, yerr=fixed_stds, marker="s", capsize=6, linewidth=2,
                color="#55A868", label="Separately-trained fixed-rank CCA")
    ax.set_xlabel("compression level (1.0 = compression=4, 0.5 = compression=8, 0.25 = compression=16)")
    ax.set_ylabel("validation loss")
    ax.set_title("The actual cost of nesting: MatCCA vs. dedicated models, at every level")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)
    print(f"  saved plot: {save_path}")


def main():
    print(f"Loading real BPE-tokenized data from '{CORPUS_PATH}' using '{TOKENIZER_PATH}'...")
    train_data, val_data, tok = prepare_dataset(CORPUS_PATH, TOKENIZER_PATH)
    train_data, val_data = train_data.to(DEVICE), val_data.to(DEVICE)
    print(f"vocab_size={tok.vocab_size}, train={len(train_data):,} tokens, "
          f"val={len(val_data):,} tokens, device={DEVICE}")

    common = dict(train_data=train_data, val_data=val_data, vocab_size=tok.vocab_size)
    summary, histories, rank_results, fixed_rank_results = {}, {}, {}, {}

    for seed in SEEDS:
        model, val_loss, hist, n_params = train_one_variant("MHA", "mha", {}, seed=seed, **common)
        summary.setdefault("MHA", []).append(val_loss)
        histories["MHA"] = hist

        _, val_loss, hist, _ = train_one_variant("GQA", "gqa", dict(n_kv_heads=2), seed=seed, **common)
        summary.setdefault("GQA", []).append(val_loss)
        histories["GQA"] = hist

        # MLA — the closest prior method to CCA, from DeepSeek-V2. Uses kv_lora_rank=DIM//4
        # to match CCA's compression factor exactly, so KV-cache sizes are directly
        # comparable. This is the comparison CCA's own paper makes, and the one the
        # reviewer report flagged as missing from our study.
        _, val_loss, hist, _ = train_one_variant(
            "MLA", "mla", dict(kv_lora_rank=DIM // 4), seed=seed, **common,
        )
        summary.setdefault("MLA", []).append(val_loss)
        histories["MLA"] = hist

        # Three SEPARATELY-TRAINED fixed-rank CCA models, one per compression level that
        # MatCCA's nested rank fractions correspond to (frac 1.0/0.5/0.25 <-> compression
        # 4/8/16). This is what actually completes the "cost of nesting" curve — before,
        # you only had the frac=1.0 anchor; now every point on MatCCA's curve has a
        # separately-trained baseline to compare against, not just the first one.
        fixed_rank_configs = {1.0: 4, 0.5: 8, 0.25: 16}
        for frac, compression in fixed_rank_configs.items():
            name = f"CCA fixed-rank (compression={compression})"
            _, val_loss, hist, _ = train_one_variant(
                name, "cca",
                dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=compression, compression_kv=compression),
                seed=seed, **common,
            )
            summary.setdefault(name, []).append(val_loss)
            histories[name] = hist
            fixed_rank_results.setdefault(frac, []).append(val_loss)

        max_r, head_dim = DIM // 4, (DIM // 4) // 8
        rank_fracs = [1.0, 0.5, 0.25]

        def sample_rank_fn():
            import random
            frac = random.choice(rank_fracs)
            r = max(head_dim, int(max_r * frac) - int(max_r * frac) % head_dim)
            return dict(active_rank_q=r, active_rank_kv=r)

        matcca_model, val_loss, hist, _ = train_one_variant(
            "MatCCA", "cca",
            dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=4, compression_kv=4),
            seed=seed, sample_rank_fn=sample_rank_fn, **common,
        )
        summary.setdefault("MatCCA", []).append(val_loss)
        histories["MatCCA"] = hist

        for frac in rank_fracs:
            r = max(head_dim, int(max_r * frac) - int(max_r * frac) % head_dim)
            rl = estimate_loss(matcca_model, val_data, BATCH_SIZE, SEQ_LEN, DEVICE, EVAL_BATCHES,
                                active_rank_q=r, active_rank_kv=r)
            rank_results.setdefault(frac, []).append(rl)

    # ---- printed summary table ----
    print(f"\n{'=' * 70}\nSUMMARY\n{'=' * 70}")
    rows = []
    stats = {}
    for name, vals in summary.items():
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if len(vals) > 1 else 0.0
        stats[name] = (vals, mean, std)
        rows.append([name, f"{mean:.4f}", f"{std:.4f}", str(len(vals))])
    print_table(rows, ["variant", "mean val loss", "std", "n seeds"])

    print("\nMatCCA by rank level (one model, evaluated at each compression):")
    rank_stats = {}
    rows2 = []
    for frac, vals in rank_results.items():
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if len(vals) > 1 else 0.0
        rank_stats[frac] = (vals, mean, std)
        rows2.append([f"{frac:.2f}", f"{mean:.4f}", f"{std:.4f}"])
    print_table(rows2, ["rank fraction", "mean val loss", "std"])

    print("\nSeparately-trained fixed-rank CCA, one model per compression level")
    print("(this is what MatCCA's curve above should be compared against — the actual")
    print("'cost of nesting' at every level, not just the frac=1.0 anchor):")
    fixed_stats = {}
    rows3 = []
    for frac, vals in fixed_rank_results.items():
        mean = statistics.mean(vals)
        std = statistics.stdev(vals) if len(vals) > 1 else 0.0
        fixed_stats[frac] = (vals, mean, std)
        rows3.append([f"{frac:.2f}", f"{mean:.4f}", f"{std:.4f}"])
    print_table(rows3, ["rank fraction", "mean val loss", "std"])

    # ---- plots ----
    print("\nGenerating plots...")
    plot_training_curves(histories, os.path.join(PLOTS_DIR, "training_curves.png"))
    plot_final_comparison(stats, os.path.join(PLOTS_DIR, "final_comparison.png"))
    plot_matcca_curve(rank_stats, os.path.join(PLOTS_DIR, "matcca_rank_curve.png"))
    plot_nesting_cost(rank_stats, fixed_stats, os.path.join(PLOTS_DIR, "nesting_cost_curve.png"))

    with open(os.path.join(PLOTS_DIR, "results_summary.json"), "w") as f:
        json.dump({
            "summary": {k: {"values": v[0], "mean": v[1], "std": v[2]} for k, v in stats.items()},
            "matcca_by_rank": {str(k): {"values": v[0], "mean": v[1], "std": v[2]} for k, v in rank_stats.items()},
            "fixed_rank_by_level": {str(k): {"values": v[0], "mean": v[1], "std": v[2]} for k, v in fixed_stats.items()},
        }, f, indent=2)
    print(f"\nAll done. Plots in '{PLOTS_DIR}/', raw numbers in '{PLOTS_DIR}/results_summary.json'.")


if __name__ == "__main__":
    main()

Overwriting train_scaled.py


In [6]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.2 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.1 min

MLA (seed=0)
  params: 26,940,416  |  device: cuda  |  seq_len: 512  |  batch: 32
  step     0 | train 330.5522 | val 271.9484 | 9s elapsed
  step   250 | train 6.9760 | val 6.9741 | 267s elapsed
  step   500 | train 5.2834 | val 5.3043 | 531s elapsed
  step   750 | train 4.7153 | val 4.7481 | 795s elapsed
  step  1000 | train 4.4871 | val 4.5668 | 1059s elapsed
  step  1250 | train 4.3854 | val 4.4415 | 1324s elapsed
  step  1500 | train 4.2632 | val 4.3186 | 1588s elapsed
  step  1750 | tr

In [3]:
!sed -i "s/SEEDS = \[0\]/SEEDS = [0, 1, 2]/" train_scaled.py
!grep "SEEDS =" train_scaled.py

Runs single-seed by default (SEEDS = [0, 1, 2]) — get this working and timed at real scale
SEEDS = [0, 1, 2]              # add more seeds here once you've timed one pass


In [ ]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.2 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.2 min

MLA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,940,416  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.4668  |  total time: 0.2 min

CCA fixed-rank (compression=4) (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.0959  |  total time: 0.2 min

CCA fixed-rank (compression=8) (seed=0)
  resumed from checkpoint at step 5000
  params: 22,260,800  |  device: cuda  |  seq_le

In [2]:
!python train_scaled.py

Loading real BPE-tokenized data from 'tinystories_train.txt' using 'tokenizer.json'...
vocab_size=8000, train=43,608,808 tokens, val=889,976 tokens, device=cuda

MHA (seed=0)
  resumed from checkpoint at step 5000
  params: 29,561,856  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2282  |  total time: 0.2 min

GQA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,416,128  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.2150  |  total time: 0.1 min

MLA (seed=0)
  resumed from checkpoint at step 5000
  params: 26,940,416  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.4668  |  total time: 0.1 min

CCA fixed-rank (compression=4) (seed=0)
  resumed from checkpoint at step 5000
  params: 23,413,824  |  device: cuda  |  seq_len: 512  |  batch: 32
  FINAL val loss: 3.0959  |  total time: 0.2 min

CCA fixed-rank (compression=8) (seed=0)
  resumed from checkpoint at step 5000
  params: 22,260,800  |  device: cuda  |  seq_le

In [3]:
!ls checkpoints_scaled/

'CCA_fixed-rank_(compression=16)_seed0.pt'   GQA_seed0.pt
'CCA_fixed-rank_(compression=16)_seed1.pt'   GQA_seed1.pt
'CCA_fixed-rank_(compression=16)_seed2.pt'   GQA_seed2.pt
'CCA_fixed-rank_(compression=4)_seed0.pt'    MatCCA_seed0.pt
'CCA_fixed-rank_(compression=4)_seed1.pt'    MatCCA_seed1.pt
'CCA_fixed-rank_(compression=4)_seed2.pt'    MatCCA_seed2.pt
'CCA_fixed-rank_(compression=8)_seed0.pt'    MHA_seed0.pt
'CCA_fixed-rank_(compression=8)_seed1.pt'    MHA_seed1.pt
'CCA_fixed-rank_(compression=8)_seed2.pt'    MHA_seed2.pt
 CCA_fixed-rank_seed0.pt		     MLA_seed0.pt
 CCA_fixed-rank_seed1.pt		     MLA_seed1.pt
 CCA_fixed-rank_seed2.pt		     MLA_seed2.pt


In [4]:
%%writefile water_filling_vs_greedy_real.py

"""
water_filling_vs_greedy_real.py — the Kaggle version: loads your actual trained MatCCA
checkpoint (from train_scaled.py) and real BPE-tokenized TinyStories data, instead of
the toy periodic-task stand-in used to verify the mechanism works at all.

Run this AFTER train_scaled.py has produced a MatCCA checkpoint (checkpoints_scaled/
MatCCA_seed0.pt or similar) — this script does not train anything itself, it only
evaluates two allocation strategies on a model you've already trained.

Cost note: real greedy search means real forward passes at real scale (dim=512,
seq_len=512, 8 layers) — expect this to take real minutes, not the ~15s the toy
version took. CALIB_BATCHES below is kept small specifically to keep this tractable;
increase it only if you have GPU-hours to spare, since every greedy step costs
CALIB_BATCHES forward passes PER CANDIDATE LAYER tried.
"""

import os
import torch

from tiny_transformer import TinyTransformerLM
from bpe_data import prepare_dataset, get_batch
from water_filling_vs_greedy import run_comparison

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CORPUS_PATH = "tinystories_train.txt"
TOKENIZER_PATH = "tokenizer.json"
CHECKPOINT_PATH = "checkpoints_scaled/MatCCA_seed0.pt"   # adjust if your seed/name differs

# Must match train_scaled.py's config exactly, or loading the checkpoint will fail.
DIM = 512
N_LAYERS = 8
N_HEADS = 8
SEQ_LEN = 512
BATCH_SIZE = 16          # smaller than training's 32 — this is just for calibration/eval
N_HEADS_LATENT = 8
COMPRESSION = 4           # the compression the checkpoint was TRAINED at (its max rank)

CALIB_BATCHES = 32         # kept small deliberately — see cost note above
COMPRESSION_FACTORS_TO_TEST = [2, 4]   # relative to the trained max rank — matches your
                                        # existing rank_fracs 0.5 and 0.25 tests, so results
                                        # are directly comparable to what you already have


def main():
    print(f"Loading real BPE-tokenized data from '{CORPUS_PATH}'...")
    train_data, val_data, tok = prepare_dataset(CORPUS_PATH, TOKENIZER_PATH)
    val_data = val_data.to(DEVICE)
    print(f"vocab_size={tok.vocab_size}, device={DEVICE}")

    max_rank = DIM // COMPRESSION
    head_dim = max_rank // N_HEADS_LATENT
    print(f"model config: max_rank={max_rank} per layer, head_dim={head_dim}, "
          f"total across {N_LAYERS} layers={N_LAYERS * max_rank}")

    print(f"\nLoading trained MatCCA checkpoint from '{CHECKPOINT_PATH}'...")
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(
            f"'{CHECKPOINT_PATH}' not found — check the exact filename in your "
            f"checkpoints_scaled/ directory (run '!ls checkpoints_scaled/' to see it) "
            f"and update CHECKPOINT_PATH above to match."
        )

    model = TinyTransformerLM(
        vocab_size=tok.vocab_size, dim=DIM, n_layers=N_LAYERS, n_heads=N_HEADS,
        max_seq_len=SEQ_LEN, attn_kind="cca",
        attn_kwargs=dict(n_heads_latent=N_HEADS_LATENT, n_kv_heads_latent=N_HEADS_LATENT,
                          compression_q=COMPRESSION, compression_kv=COMPRESSION),
    ).to(DEVICE)

    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    print(f"  loaded, checkpoint was at training step {ckpt.get('step', '?')}")
    print(f"  params: {model.param_count():,}")

    def get_batch_fn():
        return get_batch(val_data, BATCH_SIZE, SEQ_LEN, DEVICE)

    calib_batches = [get_batch_fn() for _ in range(CALIB_BATCHES)]

    results = []
    for cf in COMPRESSION_FACTORS_TO_TEST:
        r = run_comparison(model, get_batch_fn, calib_batches, N_LAYERS, head_dim, max_rank,
                            compression_factor=cf, verbose=True)
        results.append(r)

    print(f"\n{'=' * 70}\nFINAL SUMMARY — real trained model, real TinyStories data\n{'=' * 70}")
    for r in results:
        wf, gr = r["water_filling"], r["greedy"]
        verdict = "water-filling WINS on quality" if wf["loss"] < gr["loss"] else "greedy wins on quality"
        print(f"  compression {r['compression_factor']}x: "
              f"water-filling loss={wf['loss']:.4f} ({wf['time_s']:.2f}s) | "
              f"greedy loss={gr['loss']:.4f} ({gr['forward_passes']} passes, {gr['time_s']:.1f}s) "
              f"-> {verdict}")

    print("\nCompare this against your existing MatCCA-by-rank-level table from train_scaled.py")
    print("(rank 1.00/0.50/0.25 <-> the uniform allocation you already measured) — this run")
    print("tells you whether a SMARTER, heterogeneous per-layer allocation at the SAME total")
    print("budget beats the uniform one, and whether water-filling or greedy finds it better.")

    import json
    with open("wf_vs_greedy_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    print("\nSaved to 'wf_vs_greedy_results.json' — run make_paper_plots.py to generate figures.")


if __name__ == "__main__":
    main()

Overwriting water_filling_vs_greedy_real.py


In [5]:
!python water_filling_vs_greedy_real.py 

Loading real BPE-tokenized data from 'tinystories_train.txt'...
vocab_size=8000, device=cuda
model config: max_rank=128 per layer, head_dim=16, total across 8 layers=1024

Loading trained MatCCA checkpoint from 'checkpoints_scaled/MatCCA_seed0.pt'...
  loaded, checkpoint was at training step 5000
  params: 23,413,824

Compression factor 2x (target total rank: 512/1024)
Traceback (most recent call last):
  File "/kaggle/working/matcca/water_filling_vs_greedy_real.py", line 108, in <module>
    main()
  File "/kaggle/working/matcca/water_filling_vs_greedy_real.py", line 83, in main
    r = run_comparison(model, get_batch_fn, calib_batches, N_LAYERS, head_dim, max_rank,
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/kaggle/working/matcca/water_filling_vs_greedy.py", line 72, in run_comparison
    eigenvalues_per_layer = extract_eigenvalues_per_layer(model, get_batch_fn)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [7]:
import water_filling_vs_greedy
print(water_filling_vs_greedy.__file__)

/kaggle/working/matcca/water_filling_vs_greedy.py


In [8]:
!sed -n '30,40p' /kaggle/working/matcca/water_filling_vs_greedy.py

        captured = cca._captured_k_lat if source == "k" else cca._captured_q_lat
        A = torch.cat(captured, dim=0).reshape(-1, captured[0].shape[-1])
        A = A - A.mean(dim=0, keepdim=True)
        cov = (A.T @ A) / A.shape[0]
        eigvals = torch.linalg.eigvalsh(cov)
        eigvals = eigvals.flip(0).clamp(min=0).cpu().numpy()
        eigenvalues_per_layer[f"layer_{i}"] = eigvals
        cca.capture_activations = False

    return eigenvalues_per_layer



In [9]:
import water_filling_vs_greedy
import inspect

print(water_filling_vs_greedy.__file__)
print(inspect.getsource(water_filling_vs_greedy.extract_eigenvalues_per_layer))

/kaggle/working/matcca/water_filling_vs_greedy.py
def extract_eigenvalues_per_layer(model, get_batch_fn, calib_batches=20, source="k"):
    ccas = [block.attn for block in model.blocks]
    for cca in ccas:
        cca.capture_activations = True
        cca._captured_k_lat.clear()
        cca._captured_q_lat.clear()

    model.eval()
    with torch.no_grad():
        for _ in range(calib_batches):
            x, _ = get_batch_fn()
            model(x)
    model.train()

    eigenvalues_per_layer = {}
    for i, cca in enumerate(ccas):
        captured = cca._captured_k_lat if source == "k" else cca._captured_q_lat
        A = torch.cat(captured, dim=0).reshape(-1, captured[0].shape[-1])
        A = A - A.mean(dim=0, keepdim=True)
        cov = (A.T @ A) / A.shape[0]
        eigvals = torch.linalg.eigvalsh(cov)
        eigvals = eigvals.flip(0).clamp(min=0).cpu().numpy()
        eigenvalues_per_layer[f"layer_{i}"] = eigvals
        cca.capture_activations = False

    return eigenvalues

In [12]:
!grep -Rn "\.numpy()" /kaggle/working/matcca

/kaggle/working/matcca/extract_eigenvalues.py:52:        eigvals = eigvals.flip(0).clamp(min=0).numpy()  # descending, clip tiny negative numerical noise
/kaggle/working/matcca/water_filling_vs_greedy.py:35:        eigvals = eigvals.flip(0).clamp(min=0).cpu().numpy()


In [13]:
!sed -i 's/eigvals = eigvals.flip(0).clamp(min=0).numpy()/eigvals = eigvals.flip(0).clamp(min=0).detach().cpu().numpy()/g' /kaggle/working/matcca/extract_eigenvalues.py

In [14]:
!python water_filling_vs_greedy_real.py 

Loading real BPE-tokenized data from 'tinystories_train.txt'...
vocab_size=8000, device=cuda
model config: max_rank=128 per layer, head_dim=16, total across 8 layers=1024

Loading trained MatCCA checkpoint from 'checkpoints_scaled/MatCCA_seed0.pt'...
  loaded, checkpoint was at training step 5000
  params: 23,413,824

Compression factor 2x (target total rank: 512/1024)
water-filling: allocation={'layer_0': 128, 'layer_1': 128, 'layer_2': 110, 'layer_3': 65, 'layer_4': 36, 'layer_5': 21, 'layer_6': 14, 'layer_7': 10}
water-filling: wall-clock=4.453s, real calibration loss=3.2447
  greedy step 1: reduced layer 4 to rank 112 (total=1008/512, KL=0.02430, forward passes so far=72)
  greedy step 2: reduced layer 4 to rank 96 (total=992/512, KL=0.03354, forward passes so far=136)
  greedy step 3: reduced layer 4 to rank 80 (total=976/512, KL=0.06499, forward passes so far=200)
  greedy step 4: reduced layer 4 to rank 64 (total=960/512, KL=0.06486, forward passes so far=264)
  greedy step 5: r

In [5]:
!python water_filling_vs_greedy_real.py 

Loading real BPE-tokenized data from 'tinystories_train.txt'...
vocab_size=8000, device=cuda
model config: max_rank=128 per layer, head_dim=16, total across 8 layers=1024

Loading trained MatCCA checkpoint from 'checkpoints_scaled/MatCCA_seed0.pt'...
  loaded, checkpoint was at training step 5000
  params: 23,413,824

Compression factor 2x (target total rank: 512/1024)
water-filling: allocation={'layer_0': 128, 'layer_1': 128, 'layer_2': 110, 'layer_3': 65, 'layer_4': 36, 'layer_5': 21, 'layer_6': 14, 'layer_7': 10}
water-filling: wall-clock=4.917s, real calibration loss=3.2114
  greedy step 1: reduced layer 4 to rank 112 (total=1008/512, KL=0.02433, forward passes so far=288)
  greedy step 2: reduced layer 4 to rank 96 (total=992/512, KL=0.03411, forward passes so far=544)
  greedy step 3: reduced layer 4 to rank 80 (total=976/512, KL=0.06592, forward passes so far=800)
  greedy step 4: reduced layer 4 to rank 64 (total=960/512, KL=0.06669, forward passes so far=1056)
  greedy step 5:

In [15]:
%%writefile inference_profile.py

"""
inference_profile.py — measures the systems metrics that MLForSys reviewers
actually care about: tokens/second throughput, KV-cache memory footprint, and
time-to-first-token (prefill latency), across all attention variants.

IMPORTANT CONTEXT: CCA's paper claims ~1.7x prefill speedup on H100 GPUs with a
fused CUDA kernel. This script runs UNFUSED PyTorch, so the speedup will not
appear here — and that is honest, not a bug. The value of running this script is:
  1. Measuring KV-cache memory directly (not computed from formulas) — real,
     hardware-measured numbers for the paper's systems table.
  2. Showing throughput at different sequence lengths — the trend matters even
     if the absolute numbers are PyTorch-limited.
  3. Providing an honest "unfused baseline" that makes the fused-kernel claim
     from the CCA paper more, not less, credible: if unfused CCA already
     matches MHA on throughput, that's a strong result.

This script measures INFERENCE only (no backward pass), at batch_size=1
(simulating single-user serving) and batch_size=8 (simulating batched serving).
"""

import time
import torch
import gc
import json
import os

from tiny_transformer import TinyTransformerLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = "profiling_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

# Architecture configs — must match train_scaled.py exactly
DIM = 512
N_LAYERS = 8
N_HEADS = 8
VOCAB_SIZE = 8000  # from your tokenizer

VARIANTS = [
    ("MHA",       "mha", {}),
    ("GQA",       "gqa", dict(n_kv_heads=2)),
    ("MLA",       "mla", dict(kv_lora_rank=DIM // 4)),
    ("CCA 4x",    "cca", dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=4, compression_kv=4)),
    ("CCA 8x",    "cca", dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=8, compression_kv=8)),
    ("CCA 16x",   "cca", dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=16, compression_kv=16)),
    ("MatCCA",    "cca", dict(n_heads_latent=8, n_kv_heads_latent=8, compression_q=4, compression_kv=4)),
]

SEQ_LENS = [128, 256, 512]
BATCH_SIZES = [1, 8]
WARMUP_ITERS = 5
MEASURE_ITERS = 20


def measure_kv_cache_bytes(attn_kind, attn_kwargs):
    """Compute theoretical KV-cache size in bytes (fp16) per token per layer."""
    if attn_kind == "mha":
        return 2 * N_HEADS * (DIM // N_HEADS) * 2  # K+V, all heads, fp16
    if attn_kind == "gqa":
        n_kv = attn_kwargs.get("n_kv_heads", 2)
        return 2 * n_kv * (DIM // N_HEADS) * 2
    if attn_kind == "mla":
        return attn_kwargs.get("kv_lora_rank", DIM // 4) * 2
    if attn_kind == "cca":
        cq = attn_kwargs.get("compression_q", 4)
        ckv = attn_kwargs.get("compression_kv", 4)
        return 2 * (DIM // ckv) * 2  # K+V latents


@torch.no_grad()
def profile_variant(name, attn_kind, attn_kwargs, seq_len, batch_size):
    """Measure prefill throughput and latency for one (variant, seq_len, batch_size) combo."""
    torch.manual_seed(0)
    model = TinyTransformerLM(
        vocab_size=VOCAB_SIZE, dim=DIM, n_layers=N_LAYERS, n_heads=N_HEADS,
        max_seq_len=seq_len, attn_kind=attn_kind, attn_kwargs=attn_kwargs,
    ).to(DEVICE).eval()

    x = torch.randint(0, VOCAB_SIZE, (batch_size, seq_len), device=DEVICE)

    # Warmup — fills CUDA caches so timing is stable
    for _ in range(WARMUP_ITERS):
        _ = model(x)
    if DEVICE == "cuda":
        torch.cuda.synchronize()

    # Measure
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(MEASURE_ITERS):
        _ = model(x)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0

    tokens_per_sec = (batch_size * seq_len * MEASURE_ITERS) / elapsed
    ms_per_batch = (elapsed / MEASURE_ITERS) * 1000
    ttft_ms = ms_per_batch  # time-to-first-token = one full prefill pass

    # Memory
    if DEVICE == "cuda":
        mem_mb = torch.cuda.max_memory_allocated() / 1e6
        torch.cuda.reset_peak_memory_stats()
    else:
        mem_mb = 0.0

    kv_bytes_per_token_per_layer = measure_kv_cache_bytes(attn_kind, attn_kwargs)
    kv_total_mb = (kv_bytes_per_token_per_layer * seq_len * N_LAYERS * batch_size) / 1e6

    del model
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

    return {
        "tokens_per_sec": round(tokens_per_sec),
        "ttft_ms": round(ttft_ms, 2),
        "peak_gpu_mb": round(mem_mb, 1),
        "kv_cache_mb_theoretical": round(kv_total_mb, 3),
    }


def print_table(results, seq_len, batch_size):
    print(f"\n{'─'*80}")
    print(f"seq_len={seq_len}, batch_size={batch_size}, device={DEVICE}")
    print(f"{'─'*80}")
    print(f"{'Variant':<16} {'Tok/s':>10} {'TTFT (ms)':>12} {'KV-cache (MB)':>15} {'vs MHA KV':>10}")
    print(f"{'─'*80}")
    mha_kv = None
    for r in results:
        if r["name"] == "MHA":
            mha_kv = r["kv_cache_mb_theoretical"]
    for r in results:
        kv_ratio = f"{r['kv_cache_mb_theoretical'] / mha_kv:.2f}x" if mha_kv else "N/A"
        print(f"{r['name']:<16} {r['tokens_per_sec']:>10,} {r['ttft_ms']:>12.2f} "
              f"{r['kv_cache_mb_theoretical']:>15.3f} {kv_ratio:>10}")
    print(f"{'─'*80}")


def main():
    print(f"Inference profiling — device: {DEVICE}")
    print(f"Variants: {[v[0] for v in VARIANTS]}")
    print(f"Seq lens: {SEQ_LENS}, batch sizes: {BATCH_SIZES}")
    print(f"Warmup: {WARMUP_ITERS} iters, Measure: {MEASURE_ITERS} iters")
    print(f"\nNOTE: these are UNFUSED PyTorch numbers — CCA's claimed 1.7x speedup")
    print(f"requires a fused CUDA kernel (Figliolia et al. 2025) not implemented here.")
    print(f"KV-cache numbers are theoretical (computed from architecture, not measured).\n")

    all_results = []
    for seq_len in SEQ_LENS:
        for batch_size in BATCH_SIZES:
            batch_results = []
            for name, attn_kind, attn_kwargs in VARIANTS:
                print(f"  profiling {name}, seq={seq_len}, batch={batch_size}...", end=" ", flush=True)
                try:
                    stats = profile_variant(name, attn_kind, attn_kwargs, seq_len, batch_size)
                    stats.update({"name": name, "seq_len": seq_len, "batch_size": batch_size})
                    batch_results.append(stats)
                    print(f"done ({stats['tokens_per_sec']:,} tok/s)")
                except Exception as e:
                    print(f"FAILED: {e}")
                    batch_results.append({"name": name, "seq_len": seq_len,
                                          "batch_size": batch_size, "error": str(e)})
            print_table(batch_results, seq_len, batch_size)
            all_results.extend(batch_results)

    # Save for make_paper_plots.py to pick up
    out_path = os.path.join(RESULTS_DIR, "inference_profile.json")
    with open(out_path, "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"\nAll results saved to '{out_path}'")
    print("Run make_paper_plots.py after this to include these in the paper figures.")


if __name__ == "__main__":
    main()

Writing inference_profile.py


In [16]:
!python inference_profile.py

Inference profiling — device: cuda
Variants: ['MHA', 'GQA', 'MLA', 'CCA 4x', 'CCA 8x', 'CCA 16x', 'MatCCA']
Seq lens: [128, 256, 512], batch sizes: [1, 8]
Warmup: 5 iters, Measure: 20 iters

NOTE: these are UNFUSED PyTorch numbers — CCA's claimed 1.7x speedup
requires a fused CUDA kernel (Figliolia et al. 2025) not implemented here.
KV-cache numbers are theoretical (computed from architecture, not measured).

  profiling MHA, seq=128, batch=1... done (27,589 tok/s)
  profiling GQA, seq=128, batch=1... done (23,939 tok/s)
  profiling MLA, seq=128, batch=1... done (28,508 tok/s)
  profiling CCA 4x, seq=128, batch=1... done (7,602 tok/s)
  profiling CCA 8x, seq=128, batch=1... done (8,911 tok/s)
  profiling CCA 16x, seq=128, batch=1... done (9,244 tok/s)
  profiling MatCCA, seq=128, batch=1... done (7,623 tok/s)

────────────────────────────────────────────────────────────────────────────────
seq_len=128, batch_size=1, device=cuda
──────────────────────────────────────────────────────────

In [3]:
%%writefile make_paper_plots.py

"""
make_paper_plots.py — reads all results JSON files produced by train_scaled.py,
train_wikitext2.py, and water_filling_vs_greedy_real.py, and produces
publication-quality figures for the paper in one shot.

Run this AFTER all experiments finish. It reads from:
  plots/results_summary.json          (TinyStories results)
  plots_wikitext2/wikitext2_results.json  (WikiText-2 results)
  wf_vs_greedy_results.json           (water-filling comparison)

Any missing file is skipped with a warning — so you can run this incrementally
as experiments finish, not just once at the very end.
"""

import json
import os
import statistics
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

OUTPUT_DIR = "paper_plots"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- consistent color palette across all plots ----
COLORS = {
    "MHA":                          "#4C72B0",
    "GQA":                          "#DD8452",
    "MLA":                          "#8172B2",
    "CCA fixed-rank (compression=4)": "#55A868",
    "CCA fixed-rank (compression=8)": "#64B5A0",
    "CCA fixed-rank (compression=16)": "#82C099",
    "MatCCA":                       "#C44E52",
    "water-filling":                "#55A868",
    "greedy":                       "#DD8452",
    "uniform":                      "#4C72B0",
}

DISPLAY_NAMES = {
    "MHA": "MHA",
    "GQA": "GQA (4x cache)",
    "MLA": "MLA (4x cache)",
    "CCA fixed-rank (compression=4)": "CCA 4x",
    "CCA fixed-rank (compression=8)": "CCA 8x",
    "CCA fixed-rank (compression=16)": "CCA 16x",
    "MatCCA": "MatCCA (nested)",
}


def load_json(path):
    if not os.path.exists(path):
        print(f"  WARNING: '{path}' not found — skipping.")
        return None
    with open(path) as f:
        return json.load(f)


def plot_architecture_comparison(summary, dataset_label, save_path):
    """Bar chart comparing all architectures on a single dataset with error bars."""
    names = list(summary.keys())
    means = [summary[n]["mean"] for n in names]
    stds = [summary[n]["std"] for n in names]
    display = [DISPLAY_NAMES.get(n, n) for n in names]
    colors = [COLORS.get(n, "#999999") for n in names]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(display, means, yerr=stds, capsize=5, color=colors, alpha=0.85,
                   edgecolor="white", linewidth=0.5)
    for bar, mean in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                f"{mean:.3f}", ha="center", va="bottom", fontsize=8)
    ax.set_ylabel("Validation loss (lower is better)", fontsize=11)
    ax.set_title(f"Architecture comparison — {dataset_label}", fontsize=12)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {save_path}")


def plot_cross_dataset(tinystories_summary, wikitext2_summary, save_path):
    """Side-by-side comparison showing whether CCA's advantage holds across datasets."""
    # Use only the variants present in both datasets
    shared = [k for k in tinystories_summary if k in wikitext2_summary]
    x = np.arange(len(shared))
    width = 0.35

    ts_means = [tinystories_summary[k]["mean"] for k in shared]
    ts_stds = [tinystories_summary[k]["std"] for k in shared]
    wt_means = [wikitext2_summary[k]["mean"] for k in shared]
    wt_stds = [wikitext2_summary[k]["std"] for k in shared]

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - width / 2, ts_means, width, yerr=ts_stds, capsize=4,
           label="TinyStories", alpha=0.85, color="#4878CF")
    ax.bar(x + width / 2, wt_means, width, yerr=wt_stds, capsize=4,
           label="WikiText-2", alpha=0.85, color="#6ACC65")
    ax.set_xticks(x)
    ax.set_xticklabels([DISPLAY_NAMES.get(k, k) for k in shared], rotation=20)
    ax.set_ylabel("Validation loss (lower is better)", fontsize=11)
    ax.set_title("Generalization across datasets: does CCA's advantage hold?", fontsize=12)
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {save_path}")


def plot_nesting_cost_curve(matcca_by_rank, fixed_rank_by_level, save_path):
    """The core MatCCA result: one nested model vs dedicated fixed-rank models."""
    fracs = sorted(float(k) for k in matcca_by_rank.keys())
    matcca_means = [matcca_by_rank[str(f)]["mean"] for f in fracs]
    matcca_stds = [matcca_by_rank[str(f)]["std"] for f in fracs]
    fixed_means = [fixed_rank_by_level[str(f)]["mean"] for f in fracs]
    fixed_stds = [fixed_rank_by_level[str(f)]["std"] for f in fracs]

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.errorbar(fracs, matcca_means, yerr=matcca_stds, marker="o", capsize=6,
                linewidth=2.5, markersize=8, color=COLORS["MatCCA"],
                label="MatCCA (one nested model)")
    ax.errorbar(fracs, fixed_means, yerr=fixed_stds, marker="s", capsize=6,
                linewidth=2.5, markersize=8, color=COLORS["CCA fixed-rank (compression=4)"],
                label="Dedicated fixed-rank CCA (per level)")

    # Annotate the gap at each point
    for f, mm, fm in zip(fracs, matcca_means, fixed_means):
        gap = mm - fm
        ax.annotate(f"+{gap:.3f}", xy=(f, (mm + fm) / 2),
                    ha="center", fontsize=8, color="#666666")

    ax.set_xlabel("Compression level (1.0=4x, 0.5=8x, 0.25=16x)", fontsize=11)
    ax.set_ylabel("Validation loss", fontsize=11)
    ax.set_title("Cost of nesting: MatCCA vs. dedicated models at each level", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {save_path}")


def plot_water_filling_comparison(wf_results, save_path):
    """Water-filling vs greedy search vs uniform allocation."""
    factors = [r["compression_factor"] for r in wf_results]
    wf_losses = [r["water_filling"]["loss"] for r in wf_results]
    greedy_losses = [r["greedy"]["loss"] for r in wf_results]
    greedy_passes = [r["greedy"]["forward_passes"] for r in wf_results]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    # Left: quality comparison
    x = np.arange(len(factors))
    width = 0.35
    ax1.bar(x - width / 2, wf_losses, width, label="Water-filling (closed-form)",
            color=COLORS["water-filling"], alpha=0.85)
    ax1.bar(x + width / 2, greedy_losses, width, label="Greedy KL-search (expensive)",
            color=COLORS["greedy"], alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels([f"{f}x compression" for f in factors])
    ax1.set_ylabel("Calibration loss (lower is better)", fontsize=11)
    ax1.set_title("Quality: water-filling vs. greedy search", fontsize=11)
    ax1.legend(fontsize=9)
    ax1.grid(axis="y", alpha=0.3)

    # Right: cost comparison
    wf_times = [r["water_filling"]["time_s"] * 1000 for r in wf_results]  # ms
    greedy_times = [r["greedy"]["time_s"] * 1000 for r in wf_results]  # ms
    ax2.bar(x - width / 2, wf_times, width, label="Water-filling",
            color=COLORS["water-filling"], alpha=0.85)
    ax2.bar(x + width / 2, greedy_times, width, label="Greedy search",
            color=COLORS["greedy"], alpha=0.85)
    for xi, (wt, gt, gp) in enumerate(zip(wf_times, greedy_times, greedy_passes)):
        ax2.text(xi + width / 2, gt + 50, f"{gp} passes", ha="center", fontsize=8)
    ax2.set_xticks(x)
    ax2.set_xticklabels([f"{f}x compression" for f in factors])
    ax2.set_ylabel("Wall-clock time (ms)", fontsize=11)
    ax2.set_title("Cost: water-filling vs. greedy search", fontsize=11)
    ax2.legend(fontsize=9)
    ax2.grid(axis="y", alpha=0.3)

    fig.suptitle("Closed-form water-filling vs. expensive greedy KL-search", fontsize=12)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {save_path}")


def plot_inference_profile(profile_results, save_path):
    """Throughput vs KV-cache size scatter plot — the systems figure MLForSys
    reviewers expect: shows the Pareto frontier of quality vs memory tradeoff."""
    import collections
    # Group by (name, seq_len=512, batch_size=1) — the most deployment-relevant config
    by_name = collections.defaultdict(list)
    for r in profile_results:
        if "error" not in r and r.get("seq_len") == 512 and r.get("batch_size") == 1:
            by_name[r["name"]].append(r)

    if not by_name:
        # Fall back to whatever seq_len/batch_size is available
        for r in profile_results:
            if "error" not in r:
                by_name[r["name"]].append(r)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    names = list(by_name.keys())
    tok_per_sec = [by_name[n][0]["tokens_per_sec"] for n in names]
    kv_mb = [by_name[n][0]["kv_cache_mb_theoretical"] for n in names]
    ttft = [by_name[n][0]["ttft_ms"] for n in names]
    colors = [COLORS.get(n, "#999999") for n in names]

    # Left: throughput bar chart
    ax1.bar(names, tok_per_sec, color=colors, alpha=0.85, edgecolor="white")
    ax1.set_ylabel("Tokens / second (higher is better)", fontsize=11)
    ax1.set_title("Inference throughput (unfused PyTorch)", fontsize=11)
    ax1.tick_params(axis="x", rotation=25)
    ax1.grid(axis="y", alpha=0.3)
    ax1.annotate("Note: CCA speedup requires fused kernel\n(Figliolia et al. 2025)",
                 xy=(0.02, 0.97), xycoords="axes fraction", va="top", fontsize=8,
                 color="#666666")

    # Right: KV-cache size bar chart
    ax2.bar(names, kv_mb, color=colors, alpha=0.85, edgecolor="white")
    ax2.set_ylabel("KV-cache MB per sequence (lower is better)", fontsize=11)
    ax2.set_title("KV-cache memory footprint", fontsize=11)
    ax2.tick_params(axis="x", rotation=25)
    ax2.grid(axis="y", alpha=0.3)

    fig.suptitle("Inference systems metrics across attention variants", fontsize=12)
    fig.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"  saved: {save_path}")


def main():
    print("Loading results files...")
    ts = load_json("plots/results_summary.json")
    wt = load_json("plots_wikitext2/wikitext2_results.json")
    wf = load_json("wf_vs_greedy_results.json")

    if ts:
        print("\nGenerating TinyStories architecture comparison...")
        plot_architecture_comparison(
            ts["summary"], "TinyStories (BPE, 43M tokens)",
            os.path.join(OUTPUT_DIR, "fig1_tinystories_comparison.png"))

        if "fixed_rank_by_level" in ts:
            print("Generating nesting-cost curve...")
            plot_nesting_cost_curve(
                ts["matcca_by_rank"], ts["fixed_rank_by_level"],
                os.path.join(OUTPUT_DIR, "fig2_nesting_cost.png"))

    if wt:
        print("\nGenerating WikiText-2 architecture comparison...")
        plot_architecture_comparison(
            wt["summary"], "WikiText-2 (BPE, ~2M tokens)",
            os.path.join(OUTPUT_DIR, "fig3_wikitext2_comparison.png"))

    if ts and wt:
        print("Generating cross-dataset comparison...")
        plot_cross_dataset(
            ts["summary"], wt["summary"],
            os.path.join(OUTPUT_DIR, "fig4_cross_dataset.png"))

    if wf:
        print("Generating water-filling comparison...")
        plot_water_filling_comparison(
            wf, os.path.join(OUTPUT_DIR, "fig5_water_filling_vs_greedy.png"))

    profile = load_json("profiling_results/inference_profile.json")
    if profile:
        print("Generating inference profiling figure...")
        plot_inference_profile(
            profile, os.path.join(OUTPUT_DIR, "fig6_inference_profile.png"))

    print(f"\nAll plots saved to '{OUTPUT_DIR}/'")
    print("Missing any? Run the corresponding experiment first, then re-run this script.")


if __name__ == "__main__":
    main()

Writing make_paper_plots.py


In [4]:
!python make_paper_plots.py

Loading results files...

Generating TinyStories architecture comparison...
  saved: paper_plots/fig1_tinystories_comparison.png
Generating nesting-cost curve...
  saved: paper_plots/fig2_nesting_cost.png
Generating water-filling comparison...
  saved: paper_plots/fig5_water_filling_vs_greedy.png
Generating inference profiling figure...
  saved: paper_plots/fig6_inference_profile.png

All plots saved to 'paper_plots/'
Missing any? Run the corresponding experiment first, then re-run this script.


In [5]:
%%writefile model_stats_table.py

"""
model_stats_table.py — computes parameter counts and KV-cache sizes for every
attention variant, directly from architecture constants. No training needed.

This is the systematic comparison table that belongs in every efficient-attention
paper's methods section. Generates both a printed table and a saved CSV.
"""

import csv
import torch
from attention import (MultiHeadAttention, GroupedQueryAttention,
                        MultiHeadLatentAttention, CompressedConvolutionalAttention)

DIM = 512
N_HEADS = 8
SEQ_LEN = 512  # for cache size calculation


def kv_cache_bytes_per_token(attn_module, dtype_bytes=2):  # fp16 = 2 bytes
    if hasattr(attn_module, "kv_cache_dim_per_token"):
        try:
            dims = attn_module.kv_cache_dim_per_token()
        except TypeError:
            dims = attn_module.kv_cache_dim_per_token(None)
    elif isinstance(attn_module, MultiHeadAttention):
        dims = 2 * N_HEADS * (DIM // N_HEADS)  # K + V, all heads
    else:
        dims = None
    return dims * dtype_bytes if dims else None


def make_table():
    variants = [
        ("MHA",
         MultiHeadAttention(DIM, N_HEADS),
         "Full KV per head"),
        ("GQA (n_kv=2)",
         GroupedQueryAttention(DIM, N_HEADS, n_kv_heads=2),
         "Shared KV heads"),
        ("MLA (rank=128)",
         MultiHeadLatentAttention(DIM, N_HEADS, kv_lora_rank=DIM // 4),
         "Compressed latent, no FLOP saving"),
        ("CCA compression=4",
         CompressedConvolutionalAttention(DIM, N_HEADS, N_HEADS, compression_q=4, compression_kv=4),
         "Compressed latent, FLOP + cache saving"),
        ("CCA compression=8",
         CompressedConvolutionalAttention(DIM, N_HEADS, N_HEADS, compression_q=8, compression_kv=8),
         "Compressed latent, FLOP + cache saving"),
        ("CCA compression=16",
         CompressedConvolutionalAttention(DIM, N_HEADS, N_HEADS, compression_q=16, compression_kv=16),
         "Compressed latent, FLOP + cache saving"),
        ("MatCCA (nested, max=4x)",
         CompressedConvolutionalAttention(DIM, N_HEADS, N_HEADS, compression_q=4, compression_kv=4),
         "Same weights, elastic at inference"),
    ]

    rows = []
    mha_params = sum(p.numel() for p in MultiHeadAttention(DIM, N_HEADS).parameters())
    mha_cache = 2 * N_HEADS * (DIM // N_HEADS) * 2  # bytes

    print(f"\n{'Attention Architecture Comparison':^90}")
    print(f"Model dim={DIM}, n_heads={N_HEADS}, seq_len={SEQ_LEN}, fp16 (2 bytes/element)")
    print("=" * 90)
    header = f"{'Variant':<30} {'Params':>10} {'vs MHA':>8} {'KV cache/token':>16} {'vs MHA':>8}  {'Notes'}"
    print(header)
    print("-" * 90)

    for name, module, notes in variants:
        params = sum(p.numel() for p in module.parameters())
        cache_bytes = kv_cache_bytes_per_token(module)
        param_ratio = f"{params / mha_params:.2f}x"
        if cache_bytes:
            cache_ratio = f"{cache_bytes / mha_cache:.2f}x"
            cache_str = f"{cache_bytes}B"
        else:
            cache_ratio = "N/A"
            cache_str = "N/A"
        print(f"{name:<30} {params:>10,} {param_ratio:>8} {cache_str:>16} {cache_ratio:>8}  {notes}")
        rows.append({
            "variant": name, "params": params, "params_vs_mha": param_ratio,
            "kv_cache_bytes_per_token": cache_bytes or "N/A",
            "kv_cache_vs_mha": cache_ratio, "notes": notes,
        })

    print("=" * 90)

    # Save to CSV for easy import into paper
    with open("model_stats.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=rows[0].keys())
        writer.writeheader()
        writer.writerows(rows)
    print("\nSaved to 'model_stats.csv'")
    return rows


if __name__ == "__main__":
    make_table()

Writing model_stats_table.py


In [6]:

!python model_stats_table.py


                            Attention Architecture Comparison                             
Model dim=512, n_heads=8, seq_len=512, fp16 (2 bytes/element)
Variant                            Params   vs MHA   KV cache/token   vs MHA  Notes
------------------------------------------------------------------------------------------
MHA                             1,048,576    1.00x            2048B    1.00x  Full KV per head
GQA (n_kv=2)                      655,360    0.62x             512B    0.25x  Shared KV heads
MLA (rank=128)                    720,896    0.69x             256B    0.12x  Compressed latent, no FLOP saving
CCA compression=4                 280,072    0.27x             512B    0.25x  Compressed latent, FLOP + cache saving
CCA compression=8                 135,944    0.13x             256B    0.12x  Compressed latent, FLOP + cache saving
CCA compression=16                 66,952    0.06x             128B    0.06x  Compressed latent, FLOP + cache saving
MatCCA (nested, max

In [3]:
%%writefile water_filling_vs_greedy_real.py

"""
water_filling_vs_greedy_real.py — Kaggle version.

Compares three per-layer rank allocation strategies on a real trained
MatCCA checkpoint:

  1. Raw water-filling (WF):        ranks by activation eigenvalue alone
  2. Output-weighted WF (OWF):      ranks by eigenvalue * up-proj column norm^2
  3. Greedy KL-search:              iteratively minimises KL to reference model

The key question: does OWF close the quality gap between raw WF and greedy,
while still requiring zero model forward passes for the allocation step?

If OWF matches or beats greedy on quality AND requires zero forward passes,
Contribution 3 upgrades from "near-equivalent quality at lower cost" to
"equal or better quality at zero allocation cost" — a materially stronger claim.
"""

import os
import time
import json
import torch
import numpy as np

from tiny_transformer import TinyTransformerLM
from bpe_data import prepare_dataset, get_batch
from water_filling import (reverse_waterfilling_allocation,
                            output_weighted_waterfilling)
from greedy_search import greedy_rank_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
CORPUS_PATH = "tinystories_train.txt"
TOKENIZER_PATH = "tokenizer.json"
CHECKPOINT_PATH = "checkpoints_scaled/MatCCA_seed0.pt"

DIM = 512
N_LAYERS = 8
N_HEADS = 8
SEQ_LEN = 512
BATCH_SIZE = 16
N_HEADS_LATENT = 8
COMPRESSION = 4
CALIB_BATCHES = 8
COMPRESSION_FACTORS_TO_TEST = [2, 4]


# -----------------------------------------------------------------------
def load_model(tok):
    model = TinyTransformerLM(
        vocab_size=tok.vocab_size, dim=DIM, n_layers=N_LAYERS, n_heads=N_HEADS,
        max_seq_len=SEQ_LEN, attn_kind="cca",
        attn_kwargs=dict(n_heads_latent=N_HEADS_LATENT,
                         n_kv_heads_latent=N_HEADS_LATENT,
                         compression_q=COMPRESSION, compression_kv=COMPRESSION),
    ).to(DEVICE)
    if not os.path.exists(CHECKPOINT_PATH):
        raise FileNotFoundError(
            f"Checkpoint not found: '{CHECKPOINT_PATH}'. "
            f"Run: !ls checkpoints_scaled/ to see available files.")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt["model"])
    print(f"  loaded checkpoint (step {ckpt.get('step','?')}), "
          f"params: {model.param_count():,}")
    return model


# -----------------------------------------------------------------------
def extract_eigenvalues_and_norms(model, calib_batches):
    """
    Single pass over an already-sampled calibration set that collects BOTH:
      - Activation covariance eigenvalues (for raw WF)
      - Up-projection column norms squared (for OWF)
    Zero extra forward passes beyond the calibration set already in hand.
    """
    ccas = [block.attn for block in model.blocks]
    for cca in ccas:
        cca.capture_activations = True
        cca._captured_k_lat.clear()
        cca._captured_q_lat.clear()

    model.eval()
    with torch.no_grad():
        for x, _ in calib_batches:
            model(x)
    model.train()

    eigenvalues = {}
    up_norms_sq = {}
    for i, cca in enumerate(ccas):
        # --- eigenvalues from activation covariance ---
        captured = cca._captured_k_lat
        A = torch.cat(captured, dim=0).reshape(-1, captured[0].shape[-1])
        A = A - A.mean(dim=0, keepdim=True)
        cov = (A.T @ A) / A.shape[0]
        eigvals = torch.linalg.eigvalsh(cov).flip(0).clamp(min=0).cpu().numpy()
        eigenvalues[f"layer_{i}"] = eigvals

        # --- up-projection column norms (no forward pass needed) ---
        W_up = cca.up_proj.weight.detach()  # shape: (E, latent_dim_q_max)
        # column i of W_up corresponds to latent dimension i
        col_norms_sq = (W_up ** 2).sum(dim=0).cpu().numpy()  # shape: (latent_dim,)
        up_norms_sq[f"layer_{i}"] = col_norms_sq

        cca.capture_activations = False

    return eigenvalues, up_norms_sq


# -----------------------------------------------------------------------
@torch.no_grad()
def evaluate_allocation(model, calib_batches, per_layer_ranks):
    model.eval()
    losses = []
    for x, y in calib_batches:
        _, loss = model(x, targets=y, per_layer_ranks=per_layer_ranks)
        losses.append(loss.item())
    model.train()
    return sum(losses) / len(losses)


def allocation_to_per_layer_ranks(allocation, n_layers, head_dim):
    """Convert {layer_i: dims} dict to per_layer_ranks list the model expects."""
    ranks = []
    for i in range(n_layers):
        raw = allocation[f"layer_{i}"]
        rounded = max(head_dim, round(raw / head_dim) * head_dim)
        ranks.append(dict(active_rank_q=rounded, active_rank_kv=rounded))
    return ranks


def list_to_per_layer_ranks(rank_list, head_dim):
    """Convert plain list of ints (from greedy) to per_layer_ranks."""
    return [dict(active_rank_q=max(head_dim, r), active_rank_kv=max(head_dim, r))
            for r in rank_list]


# -----------------------------------------------------------------------
def run_all_comparisons(model, get_batch_fn, calib_batches,
                         n_layers, head_dim, max_rank, compression_factor):
    total_max_rank = n_layers * max_rank
    target = (total_max_rank // compression_factor // head_dim) * head_dim

    print(f"\n{'='*65}")
    print(f"Compression {compression_factor}x "
          f"(target total rank: {target}/{total_max_rank})")
    print(f"{'='*65}")

    # --- extract eigenvalues + up-proj norms in ONE pass ---
    t0 = time.time()
    eigenvalues, up_norms_sq = extract_eigenvalues_and_norms(
        model, calib_batches)
    extract_time = time.time() - t0
    print(f"Calibration pass: {extract_time:.2f}s "
          f"(shared by all three allocators)")

    # --- 1. Raw water-filling ---
    t0 = time.time()
    wf_alloc, _, _ = reverse_waterfilling_allocation(eigenvalues, target)
    wf_time = time.time() - t0
    wf_ranks = allocation_to_per_layer_ranks(wf_alloc, n_layers, head_dim)
    wf_loss = evaluate_allocation(model, calib_batches, wf_ranks)
    print(f"\nRaw WF:    alloc={[wf_alloc[f'layer_{i}'] for i in range(n_layers)]}")
    print(f"           loss={wf_loss:.4f}, sort_time={wf_time*1000:.1f}ms, "
          f"model_passes=0")

    # --- 2. Output-weighted water-filling ---
    t0 = time.time()
    owf_alloc, _ = output_weighted_waterfilling(eigenvalues, up_norms_sq, target)
    owf_time = time.time() - t0
    owf_ranks = allocation_to_per_layer_ranks(owf_alloc, n_layers, head_dim)
    owf_loss = evaluate_allocation(model, calib_batches, owf_ranks)
    print(f"\nOWF:       alloc={[owf_alloc[f'layer_{i}'] for i in range(n_layers)]}")
    print(f"           loss={owf_loss:.4f}, sort_time={owf_time*1000:.1f}ms, "
          f"model_passes=0")

    # --- 3. Greedy KL-search ---
    t0 = time.time()
    greedy_ranks_list, n_passes = greedy_rank_search(
        model, calib_batches, n_layers, head_dim, max_rank, target, verbose=False)
    greedy_time = time.time() - t0
    greedy_ranks = list_to_per_layer_ranks(greedy_ranks_list, head_dim)
    greedy_loss = evaluate_allocation(model, calib_batches, greedy_ranks)
    print(f"\nGreedy:    alloc={greedy_ranks_list}")
    print(f"           loss={greedy_loss:.4f}, time={greedy_time:.1f}s, "
          f"model_passes={n_passes}")

    # --- Summary for this compression level ---
    print(f"\n--- Verdict at {compression_factor}x ---")
    print(f"  Raw WF vs greedy:  gap={wf_loss - greedy_loss:+.4f}  "
          f"({'WF worse' if wf_loss > greedy_loss else 'WF better'})")
    print(f"  OWF vs greedy:     gap={owf_loss - greedy_loss:+.4f}  "
          f"({'OWF worse' if owf_loss > greedy_loss else 'OWF BETTER or equal'})")
    print(f"  OWF vs raw WF:     gap={owf_loss - wf_loss:+.4f}  "
          f"({'OWF worse' if owf_loss > wf_loss else 'OWF better'})")
    print(f"  Speed: WF/OWF sort={max(wf_time,owf_time)*1000:.1f}ms  "
          f"Greedy={greedy_time:.1f}s ({greedy_time/max(wf_time,owf_time,1e-9):.0f}x slower)")

    return {
        "compression_factor": compression_factor,
        "raw_wf":  {"loss": wf_loss,     "time_s": wf_time,     "passes": 0,
                    "alloc": [wf_alloc[f"layer_{i}"] for i in range(n_layers)]},
        "owf":     {"loss": owf_loss,    "time_s": owf_time,    "passes": 0,
                    "alloc": [owf_alloc[f"layer_{i}"] for i in range(n_layers)]},
        "greedy":  {"loss": greedy_loss, "time_s": greedy_time, "passes": n_passes,
                    "alloc": greedy_ranks_list},
    }


# -----------------------------------------------------------------------
def main():
    print(f"Loading data from '{CORPUS_PATH}'...")
    train_data, val_data, tok = prepare_dataset(CORPUS_PATH, TOKENIZER_PATH)
    val_data = val_data.to(DEVICE)

    max_rank = DIM // COMPRESSION
    head_dim = max_rank // N_HEADS_LATENT
    print(f"max_rank={max_rank}, head_dim={head_dim}, device={DEVICE}")

    print(f"\nLoading model from '{CHECKPOINT_PATH}'...")
    model = load_model(tok)

    def get_batch_fn():
        return get_batch(val_data, BATCH_SIZE, SEQ_LEN, DEVICE)

    calib_batches = [get_batch_fn() for _ in range(CALIB_BATCHES)]

    results = []
    for cf in COMPRESSION_FACTORS_TO_TEST:
        r = run_all_comparisons(
            model, get_batch_fn, calib_batches,
            N_LAYERS, head_dim, max_rank, cf)
        results.append(r)

    # ---- Final summary table ----
    print(f"\n{'='*65}")
    print("FINAL SUMMARY")
    print(f"{'='*65}")
    print(f"{'Compression':<14} {'Allocator':<12} {'Loss':>8} "
          f"{'vs Greedy':>12} {'Passes':>10}")
    print("-" * 65)
    for r in results:
        cf = r["compression_factor"]
        gr = r["greedy"]["loss"]
        for name, key in [("Raw WF", "raw_wf"), ("OWF", "owf"), ("Greedy", "greedy")]:
            loss = r[key]["loss"]
            gap = f"{loss - gr:+.4f}" if key != "greedy" else "---"
            passes = r[key]["passes"]
            print(f"{str(cf)+'x':<14} {name:<12} {loss:>8.4f} "
                  f"{gap:>12} {passes:>10}")
        print()

    print("Key question: does OWF close the gap to greedy while using 0 passes?")
    for r in results:
        cf = r["compression_factor"]
        wf_gap  = r["raw_wf"]["loss"] - r["greedy"]["loss"]
        owf_gap = r["owf"]["loss"]    - r["greedy"]["loss"]
        improvement = wf_gap - owf_gap
        print(f"  {cf}x: raw WF gap={wf_gap:+.4f}, OWF gap={owf_gap:+.4f}, "
              f"OWF improvement over raw WF={improvement:+.4f}")

    with open("wf_vs_greedy_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    print("\nSaved to 'wf_vs_greedy_results.json'")
    print("Run make_paper_plots.py to regenerate figures.")


if __name__ == "__main__":
    main()

Overwriting water_filling_vs_greedy_real.py


In [4]:
%%writefile water_filling.py

"""
water_filling.py — Three rank allocation strategies for MatCCA.

1. reverse_waterfilling_allocation:
   Classical rate-distortion water-filling on raw activation eigenvalues.
   O(N log N), zero model forward passes.

2. output_weighted_waterfilling:
   NEW — weights each eigenvalue by the squared L2 norm of the corresponding
   up-projection column. This captures "how much does this latent dimension
   actually affect the output" rather than just "how much variance does it
   carry in the activation space". Still O(N log N), still zero model
   forward passes. Addresses the gap between raw water-filling and greedy
   KL-search by incorporating the up-projection weight matrix as a
   first-order proxy for output importance.

   Theoretical basis: for a linear layer y = x W_up, the output variance
   contributed by latent dimension i is proportional to
   lambda_i * ||W_up[:, i]||^2. Ranking by this product instead of lambda_i
   alone gives the output-impact-aware allocation.

3. greedy_distortion_allocation:
   Cheap proxy-objective greedy (matches water-filling by construction).
   Used only for correctness verification. NOT the expensive KL-search.
"""

import numpy as np


def reverse_waterfilling_allocation(eigenvalues_per_group: dict, total_rank_budget: int):
    """
    Classical reverse water-filling on raw eigenvalues.
    Pools all (group, dimension) eigenvalues, keeps the global top-R.

    Args:
        eigenvalues_per_group: {group_id: array-like of eigenvalues}
        total_rank_budget: total dimensions to keep across all groups

    Returns:
        allocation: {group_id: int dims kept}
        threshold: float water level theta
        distortion: float total discarded variance
    """
    tagged = []
    for gid, evals in eigenvalues_per_group.items():
        for v in np.asarray(evals, dtype=np.float64):
            tagged.append((float(v), gid))
    tagged.sort(key=lambda t: -t[0])

    budget = min(total_rank_budget, len(tagged))
    kept = tagged[:budget]
    discarded = tagged[budget:]

    threshold = kept[-1][0] if kept else float("inf")
    distortion = sum(v for v, _ in discarded)

    allocation = {gid: 0 for gid in eigenvalues_per_group}
    for v, gid in kept:
        allocation[gid] += 1

    return allocation, threshold, distortion


def output_weighted_waterfilling(
    eigenvalues_per_group: dict,
    up_proj_col_norms_sq_per_group: dict,
    total_rank_budget: int,
):
    """
    Output-importance-weighted reverse water-filling.

    Scores each (group, dimension) pair by:
        score_i = lambda_i * ||W_up[:, i]||^2

    where lambda_i is the activation eigenvalue and ||W_up[:, i]||^2 is the
    squared L2 norm of column i in the up-projection weight matrix.

    This score approximates the output variance contributed by dimension i
    under a linear approximation: Var[y_i] ~ lambda_i * ||W_up[:, i]||^2.
    Keeping the top-R by this score minimises expected output distortion
    more accurately than keeping by eigenvalue alone.

    Still O(N log N). Still zero model forward passes.
    The up-projection norms come directly from the weight matrix, no
    calibration data needed.

    Args:
        eigenvalues_per_group: {group_id: array of eigenvalues, descending}
        up_proj_col_norms_sq_per_group: {group_id: array of ||W_up[:,i]||^2}
            Must have the same length as eigenvalues_per_group[group_id].
        total_rank_budget: total dimensions to keep

    Returns:
        allocation: {group_id: int dims kept}
        distortion: float total discarded raw variance (for comparability)
    """
    tagged = []
    for gid in eigenvalues_per_group:
        evals = np.asarray(eigenvalues_per_group[gid], dtype=np.float64)
        norms_sq = np.asarray(up_proj_col_norms_sq_per_group[gid], dtype=np.float64)
        assert len(evals) == len(norms_sq), (
            f"group {gid}: eigenvalue count {len(evals)} != "
            f"up-proj norm count {len(norms_sq)}"
        )
        for i, (ev, ns) in enumerate(zip(evals, norms_sq)):
            score = float(ev) * float(ns)
            tagged.append((score, float(ev), gid, i))

    tagged.sort(key=lambda t: -t[0])  # descending by output-importance score

    budget = min(total_rank_budget, len(tagged))
    kept_set = set(id(t) for t in tagged[:budget])

    allocation = {gid: 0 for gid in eigenvalues_per_group}
    discarded_variance = 0.0
    for i, (score, ev, gid, dim_idx) in enumerate(tagged):
        if i < budget:
            allocation[gid] += 1
        else:
            discarded_variance += ev

    return allocation, discarded_variance


def greedy_distortion_allocation(eigenvalues_per_group: dict, total_rank_budget: int):
    """
    Greedy on the raw distortion proxy. Guaranteed to match
    reverse_waterfilling_allocation exactly (both solve the same objective).
    Used only to verify correctness of the water-filling implementation.
    This is NOT the expensive KL-divergence greedy search.
    """
    remaining = []
    for gid, evals in eigenvalues_per_group.items():
        for v in np.asarray(evals, dtype=np.float64):
            remaining.append((float(v), gid))
    remaining.sort(key=lambda t: -t[0])

    budget = min(total_rank_budget, len(remaining))
    allocation = {gid: 0 for gid in eigenvalues_per_group}
    for v, gid in remaining[:budget]:
        allocation[gid] += 1
    return allocation


def _test():
    """Correctness and sanity checks for all three allocators."""
    rng = np.random.default_rng(0)
    n_layers, dims = 6, 16

    eigenvalues_per_group = {
        f"layer_{l}": np.sort(
            rng.exponential(scale=(n_layers - l), size=dims)
        )[::-1]
        for l in range(n_layers)
    }

    # Simulate up-proj column norms: deeper layers have smaller norms
    up_norms_sq = {
        f"layer_{l}": rng.uniform(0.1, 1.0, size=dims) * (1.0 / (l + 1))
        for l in range(n_layers)
    }

    total_dims = n_layers * dims
    print("=== Correctness: WF must match greedy proxy at every budget ===")
    for cf in [2, 4, 8]:
        budget = total_dims // cf
        wf_alloc, theta, wf_dist = reverse_waterfilling_allocation(
            eigenvalues_per_group, budget)
        greedy_alloc = greedy_distortion_allocation(eigenvalues_per_group, budget)
        assert wf_alloc == greedy_alloc, f"MISMATCH at budget={budget}"
        print(f"  {cf}x: WF matches greedy proxy. theta={theta:.4f}, "
              f"distortion={wf_dist:.4f}")

    print("\n=== Output-weighted WF: allocations should differ from raw WF ===")
    for cf in [2, 4, 8]:
        budget = total_dims // cf
        raw_alloc, _, _ = reverse_waterfilling_allocation(
            eigenvalues_per_group, budget)
        owf_alloc, owf_dist = output_weighted_waterfilling(
            eigenvalues_per_group, up_norms_sq, budget)
        differ = raw_alloc != owf_alloc
        print(f"  {cf}x: allocations differ={differ}, "
              f"OWF discarded_variance={owf_dist:.4f}")
        print(f"    raw WF:  {raw_alloc}")
        print(f"    OWF:     {owf_alloc}")

    print("\nAll checks passed.")


if __name__ == "__main__":
    _test()

Overwriting water_filling.py


In [5]:
!python water_filling_vs_greedy_real.py

Loading data from 'tinystories_train.txt'...
max_rank=128, head_dim=16, device=cuda

Loading model from 'checkpoints_scaled/MatCCA_seed0.pt'...
  loaded checkpoint (step 5000), params: 23,413,824

Compression 2x (target total rank: 512/1024)
Calibration pass: 2.97s (shared by all three allocators)

Raw WF:    alloc=[128, 128, 111, 65, 36, 21, 13, 10]
           loss=3.2349, sort_time=0.5ms, model_passes=0

OWF:       alloc=[128, 128, 111, 65, 35, 22, 13, 10]
           loss=3.2349, sort_time=0.8ms, model_passes=0

Greedy:    alloc=[128, 128, 128, 32, 16, 16, 32, 32]
           loss=3.2165, time=408.5s, model_passes=2032

--- Verdict at 2x ---
  Raw WF vs greedy:  gap=+0.0184  (WF worse)
  OWF vs greedy:     gap=+0.0184  (OWF worse)
  OWF vs raw WF:     gap=+0.0000  (OWF better)
  Speed: WF/OWF sort=0.8ms  Greedy=408.5s (492764x slower)

Compression 4x (target total rank: 256/1024)
Calibration pass: 1.79s (shared by all three allocators)

Raw WF:    alloc=[102, 83, 33, 16, 9, 6, 4, 3]
 

In [6]:
!sed -i 's/COMPRESSION_FACTORS_TO_TEST = \[2, 4\]/COMPRESSION_FACTORS_TO_TEST = [2, 4, 8]/' water_filling_vs_greedy_real.py
!grep "COMPRESSION_FACTORS_TO_TEST" water_filling_vs_greedy_real.py

COMPRESSION_FACTORS_TO_TEST = [2, 4, 8]
    for cf in COMPRESSION_FACTORS_TO_TEST:


In [7]:
!python water_filling_vs_greedy_real.py

Loading data from 'tinystories_train.txt'...
max_rank=128, head_dim=16, device=cuda

Loading model from 'checkpoints_scaled/MatCCA_seed0.pt'...
  loaded checkpoint (step 5000), params: 23,413,824

Compression 2x (target total rank: 512/1024)
Calibration pass: 1.95s (shared by all three allocators)

Raw WF:    alloc=[128, 128, 111, 65, 35, 21, 14, 10]
           loss=3.2363, sort_time=0.4ms, model_passes=0

OWF:       alloc=[128, 128, 111, 65, 35, 21, 13, 11]
           loss=3.2363, sort_time=0.9ms, model_passes=0

Greedy:    alloc=[128, 128, 128, 32, 16, 16, 32, 32]
           loss=3.2181, time=408.0s, model_passes=2032

--- Verdict at 2x ---
  Raw WF vs greedy:  gap=+0.0182  (WF worse)
  OWF vs greedy:     gap=+0.0182  (OWF worse)
  OWF vs raw WF:     gap=+0.0000  (OWF better)
  Speed: WF/OWF sort=0.9ms  Greedy=408.0s (469116x slower)

Compression 4x (target total rank: 256/1024)
Calibration pass: 1.79s (shared by all three allocators)

Raw WF:    alloc=[102, 83, 33, 16, 9, 6, 4, 3]
 